# 7. 频域分析与滤波

**目标**：理解频域表示，掌握频域滤波完整流程。

## 知识点清单

- 图像的傅里叶变换：`np.fft.fft2`、`cv2.dft`
- 用 OpenCV 实现傅里叶变换
- 用 Numpy 实现傅里叶变换
- 频谱中心化：`fftshift`
- 快速傅里叶变换：`np.fft.fft2`、`cv2.dft` 性能对比
- 频域滤波的基本步骤：变换 → 中心化 → 滤波 → 反中心化 → 逆变换
- 低通滤波：理想、巴特沃斯、高斯低通
- 低通滤波器的传递函数
- 频域滤波的详细步骤
- 高通滤波：理想、巴特沃斯、高斯高通
-  Laplacian 算子
- 梯度算子的传递函数
- 选择性滤波器：带阻滤波器、带通滤波器
- 陷波滤波器
- **如何找到噪声频率**
- **频域滤波的边界效应与补零**
- **向量化实现：meshgrid 替代双重循环**
- **空间域 vs 频域：如何选择**

**独立性**：依赖模块 1、6，是空间滤波的频域对应。

## 7.0 频域分析的本质：把图像拆成不同频率的波

### 内容介绍

前面第 6 章讲的都是**空间域滤波**：用一个小核在图像上滑动，做加权求和或排序取值。

这一章要换一个视角——**频域（Frequency Domain）**。

**一句话理解频域**：

> **任何一幅图像，都可以看成很多不同频率、不同振幅、不同相位的正弦波叠加而成。**

频域分析就是：**不去看每个像素是多少，而是去看这幅图里"各种频率的成分各有多少"。**

---

### 一、什么是"频率"

在图像里，**频率 = 像素值变化的快慢**。

| 频率 | 图像中的表现 | 举例 |
|------|-------------|------|
| **低频** | 像素值变化缓慢、平缓 | 大片天空、平坦墙面、柔和渐变 |
| **中频** | 像素值中等速度变化 | 物体轮廓、一般纹理 |
| **高频** | 像素值变化剧烈、跳变 | 边缘、细线、噪点、纹理细节 |

**用一行像素理解频率**：

```text
低频（缓慢变化）：  10  12  14  16  18  20  22  24
中频（中等变化）：  10  30  10  30  10  30  10  30
高频（剧烈跳变）：  10 200  10 200  10 200  10 200
```

- 第一行变化很慢 → 低频；
- 第二行来回震荡 → 中频；
- 第三行剧烈跳变 → 高频。

---

### 二、空间域与频域的对应关系

同一幅图，可以从两个角度描述：

| 角度 | 关注什么 | 表现方式 |
|------|---------|---------|
| **空间域** | 每个像素的灰度值 | 图像的像素矩阵 |
| **频域** | 各种频率成分的多少 | 频谱图 |

**两者的桥梁**：**傅里叶变换**。

```text
空间域图像  ──傅里叶变换──>  频域频谱
空间域图像  <──逆傅里叶变换──  频域频谱
```

---

### 三、空间滤波与频域滤波的对应

第 6 章的空间滤波，和第 7 章的频域滤波，本质上是**同一件事的两种做法**。

**核心定理（卷积定理）**：

> **空间域的卷积 = 频域的乘法**
> **空间域的相关 = 频域的共轭乘法**

用公式表示：

$$f(x, y) * h(x, y) \Longleftrightarrow F(u, v) \cdot H(u, v)$$

- $f(x, y)$：空间域图像；
- $h(x, y)$：空间域核（滤波器）；
- $F(u, v)$：频域图像；
- $H(u, v)$：频域滤波器（传递函数）；
- $*$：卷积；
- $\cdot$：逐元素乘法。

**这句话的含义**：

- 在空间域做卷积（核滑动 + 加权求和），
- 等价于在频域做乘法（频谱 × 传递函数）。

**举例对照**：

| 空间域滤波 | 频域滤波 | 效果 |
|-----------|---------|------|
| 均值滤波 / 高斯滤波 | 低通滤波 | 模糊、去噪 |
| Laplacian / Sobel | 高通滤波 | 锐化、边缘检测 |
| 带阻 / 陷波 | 带阻 / 陷波 | 去除周期性噪声 |

---

### 四、为什么要在频域做滤波

既然空间域已经能做滤波，为什么还要学频域？

**原因 1：频域滤波更直观**

- 空间域：设计一个核，很难直观看出它到底在"通过什么频率、抑制什么频率"；
- 频域：直接画一条"频率响应曲线"，一看就知道哪些频率被保留、哪些被抑制。

**原因 2：某些滤波在频域实现更简单**

- 比如**去除周期性噪声**（条纹、网纹）：
  - 空间域：很难设计一个核精准去掉某个频率；
  - 频域：找到频谱上对应的亮点，直接抹掉即可。

**原因 3：某些滤波在频域计算更快**

- 当空间域核很大时，直接卷积计算量巨大；
- 频域下：FFT → 乘法 → IFFT，可以利用快速傅里叶变换大幅加速。

**原因 4：频域是理解图像处理的必备视角**

- 采样定理、混叠、重建、压缩（JPEG）、水印……都离不开频域；
- 不懂频域，就很难理解这些高级话题。

---

### 五、频域滤波的基本流程（先建立整体印象）

频域滤波的完整流程是五步：

```text
① 傅里叶变换    F = FFT(f)
        ↓
② 频谱中心化    F_shift = fftshift(F)
        ↓
③ 频域滤波      G = F_shift × H
        ↓
④ 反中心化      G_unshift = ifftshift(G)
        ↓
⑤ 逆傅里叶变换  g = IFFT(G_unshift)
```

**每步在做什么**：

| 步骤 | 操作 | 目的 |
|------|------|------|
| ① 傅里叶变换 | 把图像从空间域变到频域 | 得到频谱 |
| ② 频谱中心化 | 把零频搬到图像中心 | 便于设计滤波器 |
| ③ 频域滤波 | 频谱与传递函数逐元素相乘 | 保留/抑制某些频率 |
| ④ 反中心化 | 把零频搬回左上角 | 为逆变换做准备 |
| ⑤ 逆傅里叶变换 | 把频域变回空间域 | 得到滤波后的图像 |

**关键点**：

- 步骤 ② 和 ④ 是为了**方便设计滤波器**，不是数学必需；
- 如果不做中心化，也可以设计滤波器，但需要把低频放在四个角，很不直观；
- 步骤 ③ 是**逐元素乘法**，不是卷积——这正是频域滤波比空间域卷积快的原因。

---

### 六、频谱图长什么样

**频谱图的直观理解**：

- **中心**：低频（图像的整体亮度、大结构）；
- **向外**：频率越来越高（细节、边缘、噪声）；
- **四个角**（未中心化时）：最高频；
- **亮度**：该频率成分的振幅大小。

**典型图像的频谱**：

| 图像类型 | 频谱特点 |
|---------|---------|
| 平滑图像（如天空） | 能量集中在中心，外围很暗 |
| 细节丰富图像（如草地） | 能量分布较广，外围有较多亮点 |
| 有周期性条纹 | 频谱上出现规律的亮点 |
| 有噪声 | 频谱外围出现散乱的亮点 |

**观察频谱的两个技巧**：

1. **对数变换**：频谱的动态范围很大，直接显示往往只能看到中心一点，用 $\log(1 + |F|)$ 可以看清全貌；
2. **中心化**：把零频搬到中心，低频在中间、高频在外围，符合直觉。

---

### 七、一句话总结

> **频域分析 = 换一个视角看图像：不看每个像素是多少，而看"各种频率的成分各有多少"。**
>
> - **低频** = 缓慢变化 = 整体亮度、大结构；
> - **高频** = 剧烈变化 = 边缘、细节、噪声；
> - **傅里叶变换** = 空间域 ↔ 频域的桥梁；
> - **卷积定理** = 空间域卷积 = 频域乘法；
> - **频域滤波五步** = 变换 → 中心化 → 滤波 → 反中心化 → 逆变换。
>
> 第 6 章的空间滤波和第 7 章的频域滤波，本质是同一件事的两种做法，只是视角不同。

## 7.1 图像的傅里叶变换：np.fft.fft2、cv2.dft

### 内容介绍

**傅里叶变换**是把图像从空间域变到频域的数学工具。在图像处理中，我们最常用的是**二维离散傅里叶变换（2D DFT）**。

**数学定义**：

$$F(u, v) = \sum_{x=0}^{M-1} \sum_{y=0}^{N-1} f(x, y) \cdot e^{-j 2\pi (ux/M + vy/N)}$$

- $f(x, y)$：空间域图像；
- $F(u, v)$：频域频谱；
- $M, N$：图像的高和宽；
- $u, v$：频率坐标。

**逆变换**：

$$f(x, y) = \frac{1}{MN} \sum_{u=0}^{M-1} \sum_{v=0}^{N-1} F(u, v) \cdot e^{j 2\pi (ux/M + vy/N)}$$

**关键性质**：

| 性质 | 说明 |
|------|------|
| 可逆性 | 正变换 + 逆变换 = 无损恢复 |
| 复数性 | 输出是复数，含**幅值**和**相位** |
| 周期性 | $F(u, v) = F(u + M, v) = F(u, v + N)$ |
| 共轭对称 | 实数图像的频谱满足 $F(-u, -v) = F^*(u, v)$，即频谱关于中心共轭对称 |
| 能量守恒 | 空间域能量 = 频域能量（Parseval 定理） |

**关于输出**：

- 傅里叶变换的输出是复数，用 `np.abs` 取**幅值**（该频率成分的强度），用 `np.angle` 取**相位**；
- 本章的滤波操作主要作用于幅值；相位在本章不是重点，读者只需知道它存在即可。

**快速傅里叶变换（FFT）**：

- 直接计算 DFT 的复杂度是 $O(M^2 N^2)$，太慢；
- FFT 利用对称性，把复杂度降到 $O(MN \log(MN))$；
- OpenCV 和 NumPy 提供的都是 FFT。

---

### 一、OpenCV 与 NumPy 两套实现

图像傅里叶变换有两种常用实现：

| 实现 | 函数 | 特点 |
|------|------|------|
| **NumPy** | `np.fft.fft2` | 简单、纯 Python、自动处理复数 |
| **OpenCV** | `cv2.dft` | 快、灵活、需要手动处理双通道复数 |

**两者对比**：

| 项目 | NumPy | OpenCV |
|------|-------|--------|
| 输入 | 任意尺寸 | 尺寸为 2 的幂时最快 |
| 输出 | 复数数组（complex128） | 双通道浮点数组（实部 + 虚部） |
| 中心化 | `np.fft.fftshift` | 手动实现或 `np.fft.fftshift` |
| 显示 | 直接 `np.abs` | 需 `cv2.split` + `cv2.magnitude` |
| 性能 | 较慢（纯 Python） | 较快（C++ 优化） |
| 易用性 | 高 | 中 |

**选择建议**：

- **学习/教学**：用 NumPy，代码简洁；
- **工程/性能**：用 OpenCV，尤其是图像大或需要实时处理时。

---

### 二、NumPy 实现傅里叶变换

**核心代码**：

```python
import numpy as np

F = np.fft.fft2(img)            # 二维傅里叶变换
F_shift = np.fft.fftshift(F)    # 中心化：零频搬到中心
magnitude = np.abs(F_shift)     # 幅值
spectrum = np.log(1 + magnitude)  # 对数变换，便于显示
```

**关键说明**：

- `np.fft.fft2`：输入实数图像，输出复数频谱；
- `np.fft.fftshift`：把零频从左上角搬到中心；
- `np.abs`：取幅值（复数模长）；
- `np.angle`：取相位（本章不展开，知道它存在即可）；
- `np.log(1 + |F|)`：对数压缩动态范围，否则只能看到中心一点。

**逆变换**：

```python
F_unshift = np.fft.ifftshift(F_shift)   # 反中心化
img_back = np.fft.ifft2(F_unshift)      # 逆变换
img_back = np.abs(img_back)             # 取幅值（结果可能是复数，但虚部应接近 0）
```

---

### 三、OpenCV 实现傅里叶变换

OpenCV 的 `cv2.dft` **不直接支持复数输入**，而是把复数拆成两个通道：

```python
import cv2
import numpy as np

# 1. 转 float32
img_float = np.float32(img)

# 2. 二维傅里叶变换
dft = cv2.dft(img_float, flags=cv2.DFT_COMPLEX_OUTPUT)

# 3. 中心化
dft_shift = np.fft.fftshift(dft)

# 4. 计算幅值
magnitude = cv2.magnitude(dft_shift[:, :, 0], dft_shift[:, :, 1])

# 5. 对数变换，便于显示
spectrum = np.log(1 + magnitude)
```

**关键说明**：

- `cv2.DFT_COMPLEX_OUTPUT`：输出为双通道（实部 + 虚部）；
- `dft[:, :, 0]`：实部；
- `dft[:, :, 1]`：虚部；
- `cv2.magnitude(实部, 虚部)`：计算 $\sqrt{实部^2 + 虚部^2}$；
- `np.fft.fftshift`：OpenCV 没有提供中心化函数，借用 NumPy 的。

**逆变换**：

```python
dft_unshift = np.fft.ifftshift(dft_shift)   # 反中心化
img_back = cv2.idft(dft_unshift)            # 逆变换
img_back = cv2.magnitude(img_back[:, :, 0], img_back[:, :, 1])  # 取幅值
```

---

### 四、为什么需要频谱中心化（fftshift）

**未中心化的频谱**：

- 零频（直流分量）在**左上角**；
- 低频在四个角，高频在中心；
- 这样的排列**不直观**，设计滤波器很麻烦。

**中心化后的频谱**：

- 零频在**中心**；
- 低频在中心附近，高频在外围；
- 符合"越往外频率越高"的直觉，便于设计滤波器。

**对比图示**：

```text
未中心化（fft2 输出）：        中心化后（fftshift）：
┌─────────────┐              ┌─────────────┐
│ 低频  低频  │              │ 高频  高频  │
│ 低频  低频  │              │ 高频  高频  │
└─────────────┘              └─────────────┘
（零频在左上角）              （零频在中心）
```

**关键**：

- `fftshift` 只是**重新排列**，不改变频谱内容；
- 滤波后要 `ifftshift` 还原，才能正确逆变换。

---

### 五、为什么要用对数变换显示频谱

**问题**：频谱的幅值动态范围极大。

- 零频幅值可能是几百万；
- 高频幅值可能只有几；
- 直接显示：只有中心一点亮，外围全黑。

**解决**：对数变换。

$$S(u, v) = \log(1 + |F(u, v)|)$$

- 用 $\log$ 压缩动态范围；
- 加 1 是为了避免 $\log(0)$；
- 变换后中心和外部的差异变小，都能看见。

**归一化到 0~255**：

```python
spectrum = np.log(1 + magnitude)
spectrum = cv2.normalize(spectrum, None, 0, 255, cv2.NORM_MINMAX)
spectrum = spectrum.astype(np.uint8)
```

---

### 六、性能对比

**FFT 的复杂度**：$O(N \log N)$（$N = M \times N$ 为像素总数）。

**NumPy vs OpenCV 性能**：

| 图像尺寸 | NumPy（ms） | OpenCV（ms） | 加速比 |
|---------|------------|-------------|--------|
| 256×256 | 约 5 | 约 1 | 5× |
| 512×512 | 约 25 | 约 5 | 5× |
| 1024×1024 | 约 120 | 约 25 | 5× |
| 2048×2048 | 约 600 | 约 120 | 5× |

**结论**：OpenCV 大约比 NumPy 快 5 倍左右。

**OpenCV 的特殊优化**：

- 当图像尺寸是 **2 的幂**时，`cv2.dft` 会使用最优的 FFT 算法；
- 如果尺寸不是 2 的幂，OpenCV 会退化为较慢的算法；
- 实践中可用 `cv2.getOptimalDFTSize` 获取最优尺寸，再 `copyMakeBorder` 填充。

**尺寸优化示例**：

```python
rows = cv2.getOptimalDFTSize(img.shape[0])   # 获取最优行数
cols = cv2.getOptimalDFTSize(img.shape[1])   # 获取最优列数
padded = cv2.copyMakeBorder(img, 0, rows - img.shape[0],
                            0, cols - img.shape[1],
                            cv2.BORDER_CONSTANT, value=0)
```

> **补充（FFT 为什么这么快）**：2D DFT 具有**可分离性**——可以先对每一列做 1D DFT，再对结果的每一行做 1D DFT。利用这一性质，2D DFT 可以分解为两次 1D DFT，复杂度从 $O(M^2N^2)$ 降到 $O(MN(M+N))$；再配合 FFT 算法，进一步降到 $O(MN\log(MN))$。这是 FFT 高效的根本原因，也是本章滤波操作能用频域实现的性能基础。

---

### 七、傅里叶变换的实际用途

| 用途 | 说明 |
|------|------|
| 频域滤波 | 低通、高通、带通、带阻、陷波 |
| 图像压缩 | JPEG 用 DCT（傅里叶变换的表亲） |
| 图像配准 | 相位相关法，用 FFT 快速找平移 |
| 卷积加速 | 大核卷积用 FFT 更快 |
| 周期性噪声去除 | 在频域找到亮点，直接抹掉 |
| 图像分析 | 频谱特征用于分类、识别 |

---

### 八、一句话总结

> **傅里叶变换 = 把图像从空间域变到频域，得到"各种频率成分的多少"。**
>
> - **NumPy**：`np.fft.fft2` + `np.fft.fftshift`，简单易用；
> - **OpenCV**：`cv2.dft` + `cv2.magnitude`，速度快，但需手动处理双通道；
> - **中心化**：把零频搬到中心，便于设计滤波器；
> - **对数变换**：压缩动态范围，便于显示；
> - **性能**：OpenCV 比 NumPy 快约 5 倍；
> - **用途**：频域滤波、压缩、配准、卷积加速、周期性噪声去除。

### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库
import time                                                  # 导入时间模块

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

# ============================================================
# 合成一张包含多种频率成分的测试图
# ============================================================
def make_test_image(size=256):
    """合成测试图：灰底 + 白色方块 + 黑色圆 + 细线 + 正弦条纹"""
    img = np.full((size, size), 100, dtype=np.uint8)         # 灰底
    cv2.rectangle(img, (30, 30), (100, 100), 255, -1)        # 白色方块（硬边缘）
    cv2.circle(img, (180, 70), 35, 0, -1)                    # 黑色圆（曲线边缘）
    cv2.line(img, (30, 180), (220, 180), 255, 1)             # 细白线（高频）
    cv2.line(img, (30, 200), (220, 200), 255, 1)             # 细白线（高频）
    # 添加水平正弦条纹（中频，模拟周期性纹理）
    for i in range(size):
        img[i, :] = np.clip(img[i, :].astype(np.int32) + 
                            15 * np.sin(2 * np.pi * i / 25), 0, 255).astype(np.uint8)
    return img

img_gray = make_test_image(256)                              # 合成测试图

# ---------- ① NumPy 实现 ----------
t0 = time.time()                                             # 计时开始
F_np = np.fft.fft2(img_gray)                                 # NumPy 傅里叶变换
F_np_shift = np.fft.fftshift(F_np)                           # 中心化
mag_np = np.abs(F_np_shift)                                  # 幅值
spec_np = np.log(1 + mag_np)                                 # 对数变换
spec_np = cv2.normalize(spec_np, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
t_np = time.time() - t0                                      # NumPy 耗时

# ---------- ② OpenCV 实现 ----------
t0 = time.time()                                             # 计时开始
dft = cv2.dft(np.float32(img_gray), flags=cv2.DFT_COMPLEX_OUTPUT)  # OpenCV 傅里叶变换
dft_shift = np.fft.fftshift(dft)                             # 中心化
mag_cv = cv2.magnitude(dft_shift[:, :, 0], dft_shift[:, :, 1])  # 幅值
spec_cv = np.log(1 + mag_cv)                                 # 对数变换
spec_cv = cv2.normalize(spec_cv, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
t_cv = time.time() - t0                                      # OpenCV 耗时

# ---------- ③ 逆变换验证 ----------
F_unshift = np.fft.ifftshift(F_np_shift)                     # 反中心化
img_back = np.abs(np.fft.ifft2(F_unshift))                   # 逆变换

# ---------- 可视化 ----------
plt.figure(figsize=(18, 5))                                  # 创建画布

plt.subplot(1, 3, 1)                                         # 子图 1
plt.title('合成测试图\n(灰底+方块+圆+细线+正弦条纹)')          # 标题
plt.imshow(img_gray, cmap='gray', vmin=0, vmax=255)          # 显示原图
plt.axis('off')                                              # 关闭坐标轴

plt.subplot(1, 3, 2)                                         # 子图 2
plt.title(f'NumPy 频谱 ({t_np*1000:.1f} ms)')                # 标题（含耗时）
plt.imshow(spec_np, cmap='gray')                             # 显示 NumPy 频谱
plt.axis('off')                                              # 关闭坐标轴

plt.subplot(1, 3, 3)                                         # 子图 3
plt.title(f'OpenCV 频谱 ({t_cv*1000:.1f} ms)')               # 标题（含耗时）
plt.imshow(spec_cv, cmap='gray')                             # 显示 OpenCV 频谱
plt.axis('off')                                              # 关闭坐标轴

plt.tight_layout()                                           # 自动调整间距
plt.show()                                                   # 显示图像

print(f"NumPy 耗时: {t_np*1000:.2f} ms")                    # 打印 NumPy 耗时
print(f"OpenCV 耗时: {t_cv*1000:.2f} ms")                   # 打印 OpenCV 耗时
print(f"逆变换误差: {np.mean(np.abs(img_gray - img_back)):.6f}")  # 打印逆变换误差

**图像解读**

一、三张子图分别是什么

| 子图 | 内容 | 对应代码 |
|------|------|---------|
| 左：合成测试图 | 灰底 + 白色方块 + 黑色圆 + 细线 + 正弦条纹 | `img_gray` |
| 中：NumPy 频谱 | 用 `np.fft.fft2` + `fftshift` + `log` 得到 | `F_np_shift` |
| 右：OpenCV 频谱 | 用 `cv2.dft` + `fftshift` + `cv2.magnitude` + `log` 得到 | `dft_shift` |

---

二、坐标轴：频谱图的定位参考系

在深入解读频谱图之前，必须先明确它的**坐标轴**。代码显示时隐藏了坐标，但读图时脑中必须要有这个坐标系。

**1. 连续数学 vs 离散实现：频谱图到底是什么？**

*   **数学理论中**：傅里叶变换的频谱是定义在连续频率域上的函数，频率坐标可以取任意实数（**包括负频率**）。理论上，它是连续的、无限精细的。
*   **计算机实现中**：我们的图像是**数字图像**（由 M × N 个像素组成的离散矩阵）。对数字图像做 DFT（离散傅里叶变换），得到的频谱**也是离散的**。
*   **你看到的频谱图**：是把这个离散矩阵用灰度图渲染出来的结果。视觉上看起来是连续的（像素很密），但本质上，它是一个由有限个矩阵元素组成的**离散像素图像**。
*   **结论**：数学上频谱是连续的，但在代码和显示中，我们处理的是离散的频谱矩阵。

**2. 为什么看不到 0 点？**

因为频谱图是一张**离散的像素图像**（矩阵），而不是一张连续的数学函数图。
*   在数学坐标系里，0 是一个无限小的点，可以用一条线标出来。
*   但在频谱图里，**正中心的那一个像素**，就是 0 点。它不是一个空心的坐标原点，而是一个实实在在的、有亮度的方块（通常是最亮的那个方块）。
*   中心那个最亮的点，它的坐标就是 `(rows//2, cols//2)`，这是一个**具体的整数索引**，而不是一个抽象的原点。

**3. 横轴（u）：水平频率**

*   代表原图**水平方向**上像素值变化的快慢。
*   从中心向左/向右，水平频率逐渐升高。
*   横轴正方向（向右）：代表水平向右传播的波纹（正频率）。
*   横轴负方向（向左）：代表水平向左传播的波纹（负频率）。

**4. 纵轴（v）：垂直频率**

*   代表原图**垂直方向**上像素值变化的快慢。
*   从中心向上/向下，垂直频率逐渐升高。
*   纵轴正方向（向下）：代表垂直向下传播的波纹（正频率）。
*   纵轴负方向（向上）：代表垂直向上传播的波纹（负频率）。

**5. 原点 (0, 0)：零频（直流分量）**

*   经过 `fftshift` 后，原点位于频谱图的**正中心那一个像素**。
*   频率为 0，代表整张图像的平均亮度。
*   原点的亮度 = 全图像素值的总和（或平均值）。

**6. 距离 D：频率高低的标尺**

*   频谱图上任意一点到中心的距离，代表该点的**频率大小**。

$$
D(u, v) = \sqrt{u^2 + v^2}
$$

*   D 越小（靠近中心），频率越低。
*   D 越大（远离中心），频率越高。
*   这正是为什么可以用“圆形”来设计低通/高通滤波器的原因——圆形边界就是等频率线。

**7. 灰度值：振幅（能量）**

*   频谱图不是三维图，横纵轴之外没有第三根坐标轴。
*   每个坐标点的**亮度（灰度值）** 就代表该频率的**振幅（能量）**。
*   越白 = 该频率能量越强；越黑 = 该频率能量越弱。

**8. 对称性：滤波必须成对处理的原因**

*   以中心 (0, 0) 像素为对称点，横轴、纵轴、斜向都是对称的。
*   中心右侧（u > 0）是正频率，左侧（u < 0）是负频率；中心下侧（v > 0）是正频率，上侧（v < 0）是负频率。它们成对出现，能量完全相等。
*   这是**实数图像傅里叶变换的固有性质（共轭对称）**：$F(-u,-v) = F^*(u,v)$。
*   **实际应用中的关键**：在设计频域滤波器（如陷波滤波器去噪）时，**必须成对处理**。如果你只在中心右侧抹掉一个亮点，而不抹掉左侧对称的亮点，就会破坏共轭对称性，逆变换出来的图像会出现错误（变成复数，无法显示）。

---

三、结合坐标轴，真正读懂这张合成图频谱

有了坐标系，我们现在可以精准地“按图索骥”。

**1. 原点 (0, 0) 极亮 —— 原图大面积平滑**

*   **定位**：频谱图的正中心那一个像素（最亮的白点）。
*   **含义**：频率为 0 的直流分量能量极强。
*   **对照原图**：灰底（100）和大面积平滑区域，整体亮度均匀，能量几乎全部集中在低频。
*   **现象**：中心有一个极其耀眼的白点。

**2. 沿纵轴 (v 轴) 的亮线 —— 原图的水平边缘**

*   **定位**：从中心沿垂直方向向外延伸的亮线（v 轴正负方向都有）。
*   **含义**：垂直方向上的中高频分量很强。
*   **对照原图**：正弦条纹是水平方向的周期性变化，因此能量集中在垂直频率轴上。
*   **规律**：**原图中的水平条纹 → 频谱图的垂直亮线（成对亮点）**。

**3. 正弦条纹的频谱特征 —— 一对亮点**

*   **定位**：纵轴上，距中心一定距离处的一对亮点。
*   **含义**：正弦条纹的周期为 25 像素，对应频率为 256/25 ≈ 10.24。
*   **现象**：在纵轴上，距离中心约 10 个像素处出现一对对称亮点。
*   **验证**：这对亮点正是陷波滤波器要抹掉的目标。

**4. 沿横轴的亮线 —— 原图的垂直边缘**

*   **定位**：从中心沿水平方向向外延伸的亮线（u 轴正负方向都有）。
*   **含义**：水平方向上的中高频分量很强。
*   **对照原图**：方块的垂直边、圆的左右边缘。
*   **规律**：**原图中的垂直边缘 → 频谱图的水平亮线**。

**5. 斜向放射线 —— 原图的斜向和曲线边缘**

*   **定位**：从中心沿对角线方向延伸的短线。
*   **含义**：斜向和曲线方向的中高频分量。
*   **对照原图**：圆的圆弧边缘、方块的角。
*   **规律**：**原图哪个方向有边缘，频谱图就在垂直于该方向的位置出现亮线**。

---

四、两图对比：NumPy 与 OpenCV

**1. 视觉上完全一致**

中图（NumPy）和右图（OpenCV）在视觉上没有任何差异。这证明了 `np.fft.fft2` 和 `cv2.dft` 在数学上完全等价。

**2. 性能差异**

看图片标题的耗时数据：OpenCV 通常比 NumPy 快数倍。虽然频谱一样，但 OpenCV 的底层 C++ 优化让它在处理大图时优势明显。

**3. 对数变换的必要性**

*   如果不做对数变换，由于低频能量太大，只有中心一点亮，外围全黑；
*   对数变换压缩了动态范围，中心和外部的细节都能看见。这是显示频谱的标准做法。

---

五、小结：带坐标系的读图四步法

> 以后拿到任何一张中心化的频谱图，脑中要建立坐标系（把最亮的中心像素当作 0 点），然后按四步读图：
>
> 1. **看原点 (0,0)**：中心越亮，说明原图大块平滑区域越多（低频强）。合成图中心极亮，灰底和平滑区域主导了能量。
> 2. **看坐标轴与放射线**：横轴亮线对应原图垂直边缘，纵轴亮线对应原图水平边缘（如正弦条纹），斜线对应斜向纹理。合成图纵轴上的成对亮点来自水平正弦条纹。
> 3. **看外围（大 D 区域）**：外围越亮、颗粒越多，说明原图细节越丰富或噪点越多。
> 4. **看对称**：检查关于中心 (0,0) 像素的对称性，确认这是实数图像的频谱。合成图完全对称，滤波时需成对处理。
>
> 这张图完美地验证了傅里叶变换的数学性质（等价性、可逆性、共轭对称性），同时也直观地展示了频域能量分布与原图空间结构的对应关系。

## 7.2 频域滤波的基本步骤

### 内容介绍

频域滤波的完整流程是五步，本小节把每步拆开讲清楚。

**完整流程**：

```text
① 傅里叶变换    F = FFT(f)
        ↓
② 频谱中心化    F_shift = fftshift(F)
        ↓
③ 频域滤波      G = F_shift × H
        ↓
④ 反中心化      G_unshift = ifftshift(G)
        ↓
⑤ 逆傅里叶变换  g = IFFT(G_unshift)
```

---

### 一、第 ① 步：傅里叶变换

**目的**：把图像从空间域变到频域。

**代码**：

```python
F = np.fft.fft2(img)            # NumPy
F = cv2.dft(np.float32(img), flags=cv2.DFT_COMPLEX_OUTPUT)  # OpenCV
```

**输出**：复数频谱 $F(u, v)$，含幅值和相位。

---

### 二、第 ② 步：频谱中心化

**目的**：把零频从左上角搬到中心，便于设计滤波器。

**代码**：

```python
F_shift = np.fft.fftshift(F)    # 中心化
```

**效果**：

- 中心 = 低频；
- 外围 = 高频；
- 符合直觉，便于设计滤波器。

---

### 三、第 ③ 步：频域滤波

**目的**：用传递函数 $H(u, v)$ 修改频谱。

**代码**：

```python
G = F_shift * H                 # 逐元素乘法
```

**传递函数 $H(u, v)$**：

- 低通滤波器：中心保留，外围抑制；
- 高通滤波器：中心抑制，外围保留；
- 带通/带阻：保留/抑制某个频率范围。

**关键**：

- 频域滤波是**逐元素乘法**，不是卷积；
- 这正是频域滤波比空间域卷积快的原因；
- 滤波器必须**共轭对称**，否则逆变换结果会变成复数。

---

### 四、第 ④ 步：反中心化

**目的**：把零频搬回左上角，为逆变换做准备。

**代码**：

```python
G_unshift = np.fft.ifftshift(G)  # 反中心化
```

**关键**：

- 滤波是在中心化频谱上做的；
- 逆变换要求频谱是未中心化的；
- 所以必须先反中心化。

---

### 五、第 ⑤ 步：逆傅里叶变换

**目的**：把频域变回空间域。

**代码**：

```python
g = np.abs(np.fft.ifft2(G_unshift))  # NumPy
g = cv2.idft(G_unshift)              # OpenCV
g = cv2.magnitude(g[:, :, 0], g[:, :, 1])  # 取幅值
```

**输出**：滤波后的图像 $g(x, y)$。

---

### 六、完整代码模板

**NumPy 版本（向量化，推荐）**：

```python
import numpy as np
import cv2

img = cv2.imread('image.jpg', 0)

# ① 傅里叶变换
F = np.fft.fft2(img)

# ② 中心化
F_shift = np.fft.fftshift(F)

# ③ 频域滤波（以理想低通为例，向量化）
rows, cols = img.shape
crow, ccol = rows // 2, cols // 2
D0 = 30
u, v = np.meshgrid(np.arange(rows), np.arange(cols), indexing='ij')
D = np.sqrt((u - crow)**2 + (v - ccol)**2)
H = (D <= D0).astype(np.float32)
G = F_shift * H

# ④ 反中心化
G_unshift = np.fft.ifftshift(G)

# ⑤ 逆变换
img_back = np.abs(np.fft.ifft2(G_unshift))
```

**OpenCV 版本**：

```python
import numpy as np
import cv2

img = cv2.imread('image.jpg', 0)

# ① 傅里叶变换
F = cv2.dft(np.float32(img), flags=cv2.DFT_COMPLEX_OUTPUT)

# ② 中心化
F_shift = np.fft.fftshift(F)

# ③ 频域滤波
H = np.zeros((img.shape[0], img.shape[1], 2), np.float32)
# ... 构造 H ...
G = F_shift * H

# ④ 反中心化
G_unshift = np.fft.ifftshift(G)

# ⑤ 逆变换
img_back = cv2.idft(G_unshift)
img_back = cv2.magnitude(img_back[:, :, 0], img_back[:, :, 1])
```

---

### 七、频域滤波的边界效应与补零（重要补充）

**1. 问题：循环卷积导致边界伪影**

直接对图像做 FFT → 滤波 → IFFT，等价于**循环卷积**，而不是**线性卷积**。

- **循环卷积**：图像边界和对面边界"绕回来"相连；
- **线性卷积**：边界外视为 0（或复制），不绕回。

**后果**：

- 图像左边缘的滤波器响应会"绕"到右边缘；
- 上下边缘同理；
- 表现为**边界出现伪影**（比如边缘出现亮带或暗带）。

**2. 解决：补零（Zero-Padding）**

**核心思想**：把图像填充到足够大，让循环卷积的结果在"有效区域"内等于线性卷积。

**规则**：

- 如果图像尺寸 $M \times N$，滤波器尺寸 $P \times Q$；
- 补零后尺寸至少为 $(M + P - 1) \times (N + Q - 1)$；
- 对于频域滤波，通常补到 $2M \times 2N$ 即可避免混叠。

**代码**：

```python
import numpy as np
import cv2

img = cv2.imread('image.jpg', 0)
rows, cols = img.shape

# 方法 1：补到 2 倍尺寸
padded = np.zeros((2*rows, 2*cols), dtype=np.float32)
padded[:rows, :cols] = img

# 方法 2：用 OpenCV 获取最优尺寸
opt_rows = cv2.getOptimalDFTSize(rows)
opt_cols = cv2.getOptimalDFTSize(cols)
padded = cv2.copyMakeBorder(img, 0, opt_rows - rows,
                            0, opt_cols - cols,
                            cv2.BORDER_CONSTANT, value=0)
```

**3. 滤波后裁剪**

滤波完成后，只取左上角 $M \times N$ 区域：

```python
img_filtered = img_filtered[:rows, :cols]
```

**4. 什么时候必须补零**

| 场景 | 是否补零 |
|------|---------|
| 单纯看频谱 | 不需要 |
| 频域低通/高通 | 建议补，避免边界伪影 |
| 频域卷积（大核） | **必须补**，否则结果错误 |
| 陷波去周期噪声 | 建议补，避免边界问题 |

**5. 补零的额外好处**

- **提高频率分辨率**：补零后频谱更精细；
- **避免频谱泄漏**：让周期假设更接近真实；
- **匹配 FFT 最优尺寸**：OpenCV 在 2 的幂时最快。

---

### 八、向量化实现：meshgrid 替代双重循环（重要补充）

**1. 问题：双重循环太慢**

前面构造滤波器 H 用的是：

```python
for u in range(rows):
    for v in range(cols):
        D = np.sqrt((u - crow)**2 + (v - ccol)**2)
        if D <= D0:
            H[u, v] = 1
```

对于 512×512 图像，要循环 26 万次，Python 循环非常慢。

**2. 向量化版本**

```python
u, v = np.meshgrid(np.arange(rows), np.arange(cols), indexing='ij')
D = np.sqrt((u - crow)**2 + (v - ccol)**2)
H = (D <= D0).astype(np.float32)
```

**3. 性能对比**

| 图像尺寸 | 双重循环 | 向量化 | 加速比 |
|---------|---------|--------|--------|
| 256×256 | 约 50 ms | 约 1 ms | 50× |
| 512×512 | 约 200 ms | 约 4 ms | 50× |
| 1024×1024 | 约 800 ms | 约 16 ms | 50× |

**4. 其他滤波器的向量化**

**高斯低通**：

```python
H = np.exp(-(D**2) / (2 * D0**2))
```

**巴特沃斯低通**：

```python
H = 1 / (1 + (D / D0)**(2 * n))
```

**高斯高通**：

```python
H = 1 - np.exp(-(D**2) / (2 * D0**2))
```

**5. 教学建议**

- **理解原理时**：用双重循环，直观；
- **实际工程时**：用向量化，快 50 倍；
- 本讲义后面的代码为了教学清晰，仍用双重循环，但读者应知道工程中要用向量化。

---

### 九、空间域 vs 频域：如何选择（重要补充）

**1. 决策表**

| 场景 | 推荐 | 原因 |
|------|------|------|
| 小核（3×3、5×5）卷积 | **空间域** | 直接卷积比 FFT 快 |
| 大核（>15×15）卷积 | **频域** | FFT 的 $O(N\log N)$ 优于直接卷积的 $O(N \cdot k^2)$ |
| 去周期噪声 | **频域** | 陷波只在频域直观 |
| 边缘检测（Sobel、Laplacian） | **空间域** | 小核，空间域更快 |
| 图像锐化（USM） | **空间域** | 小核 |
| 图像配准（平移） | **频域** | 相位相关法 |
| 大尺度模糊 | **频域** | 大核高斯模糊，频域更快 |
| 中值滤波 | **空间域** | 非线性，频域无对应 |
| 双边滤波 | **空间域** | 非线性，频域无对应 |

**2. 复杂度对比**

**空间域卷积**：$O(M \cdot N \cdot k^2)$，$k$ 是核大小；
**频域滤波**：$O(M \cdot N \cdot \log(M \cdot N))$（含两次 FFT + 一次 IFFT）。

**临界点**：当 $k^2 > \log(MN)$ 时，频域更快。

**举例**：$512 \times 512$ 图像，$\log_2(512^2) = 18$，即 $k > 4$ 时频域开始有优势（理想情况，实际还需考虑补零后尺寸）。

**3. 实现难度**

| 项目 | 空间域 | 频域 |
|------|--------|------|
| 代码量 | 少（`cv2.filter2D`） | 多（FFT + 滤波 + IFFT） |
| 边界处理 | OpenCV 自动 | 需手动补零 |
| 可解释性 | 核直观 | 传递函数直观 |
| 灵活性 | 小核 | 任意频率响应 |

**4. 实用建议**

- **默认用空间域**：小核、非线性滤波；
- **大核或频域特定需求用频域**：大核卷积、去周期噪声、配准；
- **两者结合**：先频域粗滤，再空间域精修。

---

### 十、一句话总结

> **频域滤波五步**：变换 → 中心化 → 滤波 → 反中心化 → 逆变换。
>
> - **变换**：空间域 → 频域；
> - **中心化**：零频搬到中心，便于设计滤波器；
> - **滤波**：频谱 × 传递函数（逐元素乘法）；
> - **反中心化**：零频搬回左上角，为逆变换做准备；
> - **逆变换**：频域 → 空间域，得到滤波后的图像；
> - **补零**：避免循环卷积边界伪影，推荐补到 2 倍尺寸；
> - **向量化**：用 meshgrid 替代双重循环，快 50 倍；
> - **选择**：小核用空间域，大核或去周期噪声用频域。
>
> 记住：**频域滤波 = 在频域做乘法**，这是它比空间域卷积快的根本原因。

### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

# ============================================================
# 合成一张包含多种频率成分的测试图
# ============================================================
def make_test_image(size=256):
    """合成测试图：灰底 + 白色方块 + 黑色圆 + 细线 + 正弦条纹"""
    img = np.full((size, size), 100, dtype=np.uint8)         # 灰底
    cv2.rectangle(img, (30, 30), (100, 100), 255, -1)        # 白色方块（硬边缘）
    cv2.circle(img, (180, 70), 35, 0, -1)                    # 黑色圆（曲线边缘）
    cv2.line(img, (30, 180), (220, 180), 255, 1)             # 细白线（高频）
    cv2.line(img, (30, 200), (220, 200), 255, 1)             # 细白线（高频）
    # 添加水平正弦条纹（中频，模拟周期性纹理）
    for i in range(size):
        img[i, :] = np.clip(img[i, :].astype(np.int32) + 
                            15 * np.sin(2 * np.pi * i / 25), 0, 255).astype(np.uint8)
    return img

img_gray = make_test_image(256)                              # 合成测试图

# ① 傅里叶变换
F = np.fft.fft2(img_gray)                                    # 傅里叶变换

# ② 中心化
F_shift = np.fft.fftshift(F)                                 # 中心化

# ③ 频域滤波（以理想低通为例，D0=30，向量化）
rows, cols = img_gray.shape                                  # 图像尺寸
crow, ccol = rows // 2, cols // 2                            # 中心坐标
D0 = 30                                                      # 截止频率
u, v = np.meshgrid(np.arange(rows), np.arange(cols), indexing='ij')
D = np.sqrt((u - crow)**2 + (v - ccol)**2)                   # 到中心的距离
H = (D <= D0).astype(np.float32)                             # 理想低通 H
G = F_shift * H                                              # 频域滤波

# ④ 反中心化
G_unshift = np.fft.ifftshift(G)                              # 反中心化

# ⑤ 逆变换
img_back = np.abs(np.fft.ifft2(G_unshift))                   # 逆变换

# ---------- 可视化 ----------
plt.figure(figsize=(15, 10))                                 # 创建画布

plt.subplot(2, 3, 1)                                         # 子图 1
plt.title('合成测试图\n(灰底+方块+圆+细线+正弦条纹)')          # 标题
plt.imshow(img_gray, cmap='gray', vmin=0, vmax=255)          # 显示原图
plt.axis('off')                                              # 关闭坐标轴

plt.subplot(2, 3, 2)                                         # 子图 2
plt.title('频谱（中心化）')                                    # 标题
spec = np.log(1 + np.abs(F_shift))                           # 对数频谱
plt.imshow(spec, cmap='gray')                                # 显示频谱
plt.axis('off')                                              # 关闭坐标轴

plt.subplot(2, 3, 3)                                         # 子图 3
plt.title('传递函数 H（理想低通）')                             # 标题
plt.imshow(H, cmap='gray', vmin=0, vmax=1)                   # 显示传递函数
plt.axis('off')                                              # 关闭坐标轴

plt.subplot(2, 3, 4)                                         # 子图 4
plt.title('滤波后频谱')                                       # 标题
spec_filt = np.log(1 + np.abs(G))                            # 滤波后频谱
plt.imshow(spec_filt, cmap='gray')                           # 显示滤波后频谱
plt.axis('off')                                              # 关闭坐标轴

plt.subplot(2, 3, 5)                                         # 子图 5
plt.title('反中心化频谱')                                     # 标题
spec_unshift = np.log(1 + np.abs(G_unshift))                 # 反中心化频谱
plt.imshow(spec_unshift, cmap='gray')                        # 显示反中心化频谱
plt.axis('off')                                              # 关闭坐标轴

plt.subplot(2, 3, 6)                                         # 子图 6
plt.title('滤波后图像（理想低通）')                            # 标题
plt.imshow(img_back, cmap='gray', vmin=0, vmax=255)          # 显示滤波后图像
plt.axis('off')                                              # 关闭坐标轴

plt.tight_layout()                                           # 自动调整间距
plt.show()                                                   # 显示图像

**图像解读**

一、六张子图分别是什么

这张图完整地展示了**频域滤波（以理想低通为例）的五个核心步骤**，对应了“变换 → 中心化 → 滤波 → 反中心化 → 逆变换”的完整流程。

| 位置 | 子图 | 对应代码 | 说明 |
|------|------|---------|------|
| 上排左 | 合成测试图 | `img_gray` | 灰底+方块+圆+细线+正弦条纹 |
| 上排中 | 频谱（中心化） | `np.log(1 + np.abs(F_shift))` | 经过 `fft2` + `fftshift` 得到 |
| 上排右 | 传递函数 H（理想低通） | `H` | 滤波器的“筛子” |
| 下排左 | 滤波后频谱 | `np.log(1 + np.abs(G))` | 频谱 × 传递函数 |
| 下排中 | 反中心化频谱 | `np.log(1 + np.abs(G_unshift))` | 经过 `ifftshift` 得到 |
| 下排右 | 滤波后图像（理想低通） | `img_back` | 经过 `ifft2` 得到最终结果 |

---

二、什么是“传递函数（H）”？

你可以把传递函数 H 想象成一个**“筛子”**或者**“滤网”**。

*   在频域里，我们不是直接去改图像，而是去改**频谱**。
*   怎么改？就是用**传递函数 H** 去乘频谱。
*   **乘法规则**：
    *   如果 H 在某个位置的值是 **1（白色）**，那频谱在这个位置的信息就**原封不动地保留**。
    *   如果 H 在某个位置的值是 **0（黑色）**，那频谱在这个位置的信息就被**直接抹掉**。
    *   如果是介于 0~1 之间的灰度，那就是**按比例削弱**。

---

三、六张图的完整流转过程（真正读图）

我们按照从左到右、从上到下的顺序，一步步拆解：

**1. 合成测试图（左上）**

*   这是输入图像。包含多种频率成分：灰底（低频）、方块和圆（中高频边缘）、细线（高频）、正弦条纹（中频周期性纹理）。

**2. 频谱（中心化）（中上）**

*   经过 `fft2` + `fftshift` 得到。
*   中心最亮（低频，代表灰底），纵轴上有成对亮点（来自正弦条纹），外围有放射线（来自边缘）。
*   注意：正负频率成对出现，这是实信号频谱的共轭对称性。

**3. 传递函数 H（理想低通）（右上）**

*   这就是“传递函数”。它是一个**只有中心一个白色圆，其余全是黑色**的图。
*   **含义**：中心白色圆代表“保留低频”，外围黑色代表“全部杀掉高频”。
*   这就是“理想低通”的硬边界——频段要么全留，要么全杀。

**4. 滤波后频谱（左下）**

*   **计算**：频谱（中上） × 传递函数（右上）。
*   **现象**：原本频谱图里的放射线和纵轴上的亮点**全没了**，只剩下中心那个白色圆里的一点低频信息。
*   **说明**：高频信息已经被传递函数彻底抹杀，正弦条纹的亮点也被抹掉了。

**5. 反中心化频谱（中下）**

*   **操作**：对左下角进行 `ifftshift`。
*   **目的**：把零频从中心搬回左上角，为逆变换做准备。
*   **现象**：中心那个亮斑跑到了左上角。这看起来很奇怪，但这是逆变换的数学要求。

**6. 滤波后图像（理想低通）（右下）**

*   **操作**：对中下进行 `ifft2`，取模长，得到最终图像。
*   **现象**：图像变得**非常模糊**，细线和正弦条纹完全消失，方块和圆的边缘变得模糊。
*   **原因**：因为高频（细线、边缘）和中频（正弦条纹）全被传递函数杀掉了，只剩下大块的低频信息（灰底）。
*   **注意振铃**：如果你仔细看方块和圆的边缘，会发现有一圈一圈的波纹（光晕）。这是因为“理想低通”的传递函数是硬边界（突然从 1 变 0），在空间域会产生“振铃效应”。

---

四、深入剖析：为什么“黑到白”的地方看起来变化不大？

很多人看到右下角的图像，会有一个疑问：**“既然高频被删了，那方块和圆的边缘变化最剧烈，应该最先被删掉，为什么看起来还在？”**

这是一个非常经典的视觉错觉，需要从以下几个方面来拆解：

**1. 为什么“黑到白”是高频？**

*   在图像里，**高频 = 像素值发生剧烈跳变的地方**。
*   你看合成图里方块和圆的边缘，是从 100（灰）突然变到 255（白）或 0（黑），这是一个极其剧烈的跳变。在数学上，这包含了极其丰富的高频分量。
*   所以，按照理论，传递函数 H 应该把边缘的高频全部杀掉，结果应该是**边缘变得极其模糊**。

**2. 为什么图中方块和圆的轮廓依然可辨认？**

*   如果你仔细看右下角的“滤波后图像”，方块和圆确实还在，但它已经**不是原来的样子了**。
*   **原图的边缘**：锐利，黑白分明，像刀切一样。
*   **滤波后的边缘**：变得**模糊、毛糙**，边缘附近出现了一圈灰色的过渡带（因为高频被删了，黑白之间的跳变变成了平滑渐变）。
*   **所以，高频确实被删掉了。** 只是因为灰底和方块内部的大块低频信息依然保留了下来，所以你视觉上还能认出“这里有个方块”。

**3. 真正的视觉错觉：为什么觉得“变化不大”？**

*   因为人眼对**低频（大块颜色）**非常敏感，对**高频（边缘锐度）**相对不敏感。
*   当你看到右下角图像时，你的大脑首先捕捉到的是“哦，这里有个方块”，而不是“哦，这个方块的边缘锐度下降了”。
*   如果你把原图和右下角的图**并排放大看边缘**，你会立刻发现：原图的边缘是“一刀切”，滤波后的边缘是“晕开”的。

**4. 那为什么细线和正弦条纹“消失”得那么明显？**

*   因为细线**本身就是由高频细节构成的**，正弦条纹的频率也高于截止频率 D0=30。
*   细线：只有 1 像素宽，像素值不断快速跳变，全是高频。
*   正弦条纹：周期 25 像素，频率约 10，D0=30 时理论上应保留基频，但理想低通会把它周围的高频谐波抹掉，导致条纹变得模糊。
*   这些细节**没有大块的低频支撑**，一旦高频被删，它们就直接变成了一片模糊的灰色，所以你看上去觉得“它们消失了”。

**5. 总结：高频被删了，但低频还在**

*   **边缘**：高频，被删了。所以边缘变模糊了。
*   **方块/圆内部**：低频（大块颜色），保留了。
*   **细线/条纹**：全是高频或中高频，被删了。所以直接消失了。
*   **灰底**：低频，保留了。

所以，右下角的图像之所以看起来“变化不大”，是因为**低频（灰底和大块颜色）还在**，你还能认出这是方块和圆。但如果你仔细看细节，你会发现所有**尖锐的边缘**都被磨圆了，这就是高频被删掉的直接证据。

---

五、一句话总结

> **传递函数 H 就是一把“频率剪刀”。**
>
> *   它决定了哪些频率留下，哪些频率剪掉。
> *   右上角的“理想低通 H”，就是一把只留中心（低频）、剪掉外围（高频）的剪刀。
> *   经过这把剪刀剪过的频谱（左下），高频全丢，逆变换回空间域（右下）后，图像就变成了模糊版。
>
> 如果把“理想低通”换成“高斯低通”，传递函数 H 的边缘会变得柔和（不会突然从 1 变 0），那么最终右下角的图像就不会有明显的振铃，模糊过渡会更自然。

## 7.3 低通滤波：理想、巴特沃斯、高斯低通

### 内容介绍

**低通滤波（Low-Pass Filter, LPF）**：保留低频，抑制高频。

**效果**：图像变模糊，去噪，平滑。

**三种常见低通滤波器**：

| 滤波器 | 传递函数 | 特点 |
|--------|---------|------|
| **理想低通** | $H(u,v) = \begin{cases} 1, & D(u,v) \le D_0 \\ 0, & D(u,v) > D_0 \end{cases}$ | 硬边界，振铃明显 |
| **巴特沃斯低通** | $H(u,v) = \frac{1}{1 + [D(u,v)/D_0]^{2n}}$ | 软边界，振铃较小 |
| **高斯低通** | $H(u,v) = e^{-D^2(u,v)/(2D_0^2)}$ | 最平滑，几乎无振铃 |

- $D(u, v)$：到频谱中心的距离；
- $D_0$：截止频率；
- $n$：巴特沃斯阶数，$n$ 越大越接近理想低通。

---

### 一、为什么叫"低通"

**低通** = **低频通过，高频被抑制**。

在频谱图上：

- **中心** = 低频 → 保留；
- **外围** = 高频 → 抑制。

**传递函数 H 的图形**：

```text
H(u,v)
  1 │━━━━━━━━
    │
  0 │        ━━━━━━━━
    └────────────────→ D
            D0
```

- $D < D_0$：$H = 1$（通过）；
- $D > D_0$：$H = 0$（抑制）；
- $D_0$：截止频率，决定保留多少低频。

**注意**：传递函数 $H$ 在正负频率上是对称的，所以上述图形同时适用于 $D$ 和 $-D$。

---

### 二、三种低通滤波器对比

**1. 理想低通（Ideal LPF）**

- 传递函数：硬边界，$D_0$ 内全 1，外全 0；
- 空间域对应：sinc 函数；
- 缺点：**振铃效应**明显（边缘附近出现波纹）；
- 实际中很少直接用。

**2. 巴特沃斯低通（Butterworth LPF）**

- 传递函数：平滑过渡，$D = D_0$ 时 $H = 0.5$；
- 阶数 $n$ 控制过渡带宽度：
  - $n = 1$：过渡很宽，很平滑；
  - $n$ 增大：逐渐接近理想低通；
  - $n \to \infty$：变成理想低通；
- 优点：振铃比理想低通小，过渡可调；
- 常用 $n = 2$ 或 $n = 3$。

**3. 高斯低通（Gaussian LPF）**

- 传递函数：高斯函数，最平滑；
- 空间域对应：高斯核（6.4.2 节讲过）；
- 优点：**几乎无振铃**，过渡最自然；
- 缺点：过渡带较宽，截止频率定义不如理想低通明确；
- 实际中最常用。

**三者对比**：

| 滤波器 | 过渡带 | 振铃 | 空间域核 |
|--------|--------|------|---------|
| 理想 | 无（突变） | 明显 | sinc（无限宽） |
| 巴特沃斯 | 可调 | 较小 | 介于两者之间 |
| 高斯 | 宽 | 几乎无 | 高斯（有限宽） |

---

### 三、低通滤波的效果

**视觉效果**：

- 图像变模糊；
- 高频细节（毛发、纹理、噪声）被抹掉；
- 大结构（轮廓、整体亮度）保留。

**截止频率 $D_0$ 的影响**：

| $D_0$ | 效果 |
|-------|------|
| 很小（如 10） | 严重模糊，只剩大块颜色 |
| 中等（如 30） | 适度模糊，细节丢失但轮廓保留 |
| 很大（如 100） | 轻微模糊，接近原图 |

**典型用途**：

- 去噪：高频噪声被抑制；
- 平滑：图像预处理；
- 压缩：丢掉高频，减少数据量。

---

### 四、低通滤波的传递函数

**理想低通**：

$$H(u, v) = \begin{cases} 1, & D(u, v) \le D_0 \\ 0, & D(u, v) > D_0 \end{cases}$$

**巴特沃斯低通**：

$$H(u, v) = \frac{1}{1 + [D(u, v)/D_0]^{2n}}$$

**高斯低通**：

$$H(u, v) = e^{-D^2(u, v)/(2D_0^2)}$$

其中：

$$D(u, v) = \sqrt{(u - M/2)^2 + (v - N/2)^2}$$

- $M, N$：图像尺寸；
- $(M/2, N/2)$：频谱中心；
- $D(u, v)$：到中心的距离。

**关键**：这些传递函数都是**径向对称**的，且关于 0 Hz 对称，所以负频率和正频率被同等对待——符合共轭对称要求。

---

### 五、代码实现模板（向量化）

```python
import numpy as np
import cv2

def make_distance_grid(shape):
    """构造到频谱中心的距离矩阵 D，向量化"""
    rows, cols = shape
    crow, ccol = rows // 2, cols // 2
    u, v = np.meshgrid(np.arange(rows), np.arange(cols), indexing='ij')
    D = np.sqrt((u - crow)**2 + (v - ccol)**2)
    return D

def ideal_lowpass(shape, D0):
    D = make_distance_grid(shape)
    return (D <= D0).astype(np.float32)

def butterworth_lowpass(shape, D0, n=2):
    D = make_distance_grid(shape)
    return 1 / (1 + (D / D0)**(2 * n))

def gaussian_lowpass(shape, D0):
    D = make_distance_grid(shape)
    return np.exp(-(D**2) / (2 * D0**2))
```

---

### 六、一句话总结

> **低通滤波 = 保留低频，抑制高频 = 图像变模糊。**
>
> - **理想低通**：硬边界，振铃明显；
> - **巴特沃斯低通**：软边界，阶数可调，振铃较小；
> - **高斯低通**：最平滑，几乎无振铃，最常用；
> - **截止频率 $D_0$**：越大保留越多，图像越清晰；越小越模糊。
>
> 空间域的高斯滤波（6.4.2 节）和频域的高斯低通，本质是同一件事。

### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

# ============================================================
# 1. 合成一张"理想测试图"
# ============================================================
def make_test_image(size=256):
    """合成测试图：灰底 + 白色方块 + 黑色圆 + 细线 + 噪点"""
    img = np.full((size, size), 128, dtype=np.uint8)         # 灰底 128
    cv2.rectangle(img, (30, 30), (100, 100), 255, -1)        # 白色方块（硬边缘）
    cv2.circle(img, (180, 70), 35, 0, -1)                    # 黑色圆（曲线边缘）
    cv2.line(img, (30, 180), (220, 180), 255, 1)             # 细白线（高频）
    cv2.line(img, (30, 200), (220, 200), 255, 1)             # 细白线（高频）
    # 添加随机噪点（高频）
    noise = np.random.randint(0, 50, (size, size), dtype=np.uint8)
    img = np.clip(img.astype(np.int32) + noise - 25, 0, 255).astype(np.uint8)
    return img

# ============================================================
# 2. 三种低通滤波器（向量化）
# ============================================================
def make_distance_grid(shape):
    rows, cols = shape
    crow, ccol = rows // 2, cols // 2
    u, v = np.meshgrid(np.arange(rows), np.arange(cols), indexing='ij')
    return np.sqrt((u - crow)**2 + (v - ccol)**2)

def ideal_lowpass(shape, D0):
    D = make_distance_grid(shape)
    return (D <= D0).astype(np.float32)

def butterworth_lowpass(shape, D0, n=2):
    D = make_distance_grid(shape)
    return 1 / (1 + (D / D0)**(2 * n))

def gaussian_lowpass(shape, D0):
    D = make_distance_grid(shape)
    return np.exp(-(D**2) / (2 * D0**2))

# ============================================================
# 3. 合成图 → 频域滤波
# ============================================================
img_gray = make_test_image(256)                              # 合成测试图
F = np.fft.fft2(img_gray)                                    # 傅里叶变换
F_shift = np.fft.fftshift(F)                                 # 中心化

D0 = 30                                                      # 截止频率
H_ideal = ideal_lowpass(img_gray.shape, D0)                  # 理想低通 H
H_butter = butterworth_lowpass(img_gray.shape, D0, n=2)      # 巴特沃斯低通 H
H_gauss = gaussian_lowpass(img_gray.shape, D0)               # 高斯低通 H

def apply_filter(F_shift, H):
    G = F_shift * H
    G_unshift = np.fft.ifftshift(G)
    return np.abs(np.fft.ifft2(G_unshift))

img_ideal = apply_filter(F_shift, H_ideal)                   # 理想低通结果
img_butter = apply_filter(F_shift, H_butter)                 # 巴特沃斯低通结果
img_gauss = apply_filter(F_shift, H_gauss)                   # 高斯低通结果

# ============================================================
# 4. 可视化（第二行整体右移一格）
# ============================================================
plt.figure(figsize=(16, 10))

# ---------- 第一行：原图 + 三种传递函数 ----------
plt.subplot(2, 4, 1)
plt.title('合成测试图（灰底 + 方块 + 圆 + 细线 + 噪点）')
plt.imshow(img_gray, cmap='gray', vmin=0, vmax=255)
plt.axis('off')

plt.subplot(2, 4, 2)
plt.title('理想低通 H')
plt.imshow(H_ideal, cmap='gray', vmin=0, vmax=1)
plt.axis('off')

plt.subplot(2, 4, 3)
plt.title('巴特沃斯低通 H (n=2)')
plt.imshow(H_butter, cmap='gray', vmin=0, vmax=1)
plt.axis('off')

plt.subplot(2, 4, 4)
plt.title('高斯低通 H')
plt.imshow(H_gauss, cmap='gray', vmin=0, vmax=1)
plt.axis('off')

# ---------- 第二行：三种滤波结果（整体右移一格，占据第 2、3、4 列）----------
plt.subplot(2, 4, 6)                                         # 第 2 列
plt.title('理想低通结果（振铃明显）')
plt.imshow(img_ideal, cmap='gray', vmin=0, vmax=255)
plt.axis('off')

plt.subplot(2, 4, 7)                                         # 第 3 列
plt.title('巴特沃斯低通结果（振铃较小）')
plt.imshow(img_butter, cmap='gray', vmin=0, vmax=255)
plt.axis('off')

plt.subplot(2, 4, 8)                                         # 第 4 列
plt.title('高斯低通结果（几乎无振铃）')
plt.imshow(img_gauss, cmap='gray', vmin=0, vmax=255)
plt.axis('off')

plt.tight_layout()
plt.show()

**图像解读**

一、七张子图分别是什么

| 位置 | 子图 | 说明 |
|------|------|------|
| 第一行第 1 列 | 合成测试图 | 包含灰底、白色方块、黑色圆、细白线、随机噪点 |
| 第一行第 2 列 | 理想低通 H | 硬边界，中心白圆，外围全黑 |
| 第一行第 3 列 | 巴特沃斯低通 H (n=2) | 软边界，中心白，向外平滑过渡到黑 |
| 第一行第 4 列 | 高斯低通 H | 最平滑，中心白，向外高斯衰减到黑 |
| 第二行第 1 列 | 理想低通结果 | 边缘出现明显**振铃**，细线出现重影 |
| 第二行第 2 列 | 巴特沃斯低通结果 | 边缘振铃明显减小，细线仍有轻微重影 |
| 第二行第 3 列 | 高斯低通结果 | 边缘最平滑，**几乎无振铃**，细线重影最弱 |

> **注**：第二行整体右移了一格（占据了第 2、3、4 列），这是为了保持与第一行的列对齐，视觉上更平衡。

---

二、为什么这张合成图最典型？

因为这张图**同时包含了三种关键的频率成分**，能一眼看出三种滤波器的差异：

| 原图元素 | 对应的频率成分 | 在三种低通中的表现 |
|---------|--------------|------------------|
| **灰底（128）** | 低频（直流分量） | 三种低通都**完整保留**灰底，亮度基本不变 |
| **白色方块** | 硬边缘，高频丰富 | 理想低通：边缘**振铃明显**（波纹）；巴特沃斯：振铃较小；高斯：几乎无振铃 |
| **黑色圆** | 曲线边缘，中高频 | 同上，圆弧边缘的振铃最容易被看出 |
| **细白线** | 极高频（接近截止频率） | 三种低通都会**把细线抹掉**或**留下重影**，但程度不同 |
| **随机噪点** | 极高频 | 三种低通都**去噪**，但理想低通会留下振铃伪影 |

---

三、逐图对比解读

**1. 三种传递函数 H**

| H | 特点 |
|---|------|
| 理想低通 | 中心白色圆，边界锐利，外围纯黑。H 在 D0 处从 1 突变为 0。 |
| 巴特沃斯低通 | 中心白色，边缘平滑过渡，外围灰色渐变。H 在 D0 处约为 0.5。 |
| 高斯低通 | 中心白色，过渡最平滑。H 随 D 增大而高斯衰减，没有明确的截止边界。 |

**2. 三种滤波结果**

| 结果 | 振铃 | 细线 | 噪点 | 整体观感 |
|------|------|------|------|---------|
| **理想低通** | **明显**（方块和圆边缘有一圈圈波纹） | 明显重影 | 去噪但留下振铃伪影 | 不自然，有明显伪影 |
| **巴特沃斯低通** | **较小**（边缘波纹减弱） | 轻微重影 | 去噪较干净 | 较自然，伪影较少 |
| **高斯低通** | **几乎无**（边缘最平滑） | 最弱重影 | 去噪干净 | 最自然，无伪影 |

---

四、深入剖析：为什么“理想低通”反而有一圈圈波纹？

**1. 先明确：“理想低通”的“理想”指什么？**

这里的“理想”，**不是指“视觉效果理想”，而是指“数学定义理想”**。

| 角度 | 含义 |
|------|------|
| **数学上的理想** | 传递函数是完美的矩形：D0 以内全通过（H=1），D0 以外全抑制（H=0），边界是**突变的、无限陡峭的** |
| **工程上的“理想”** | 实际用起来效果很差，因为会产生严重的振铃伪影 |

所以，“理想”是一个**数学术语**——它完美符合“低通”的数学定义，但**不实用**。

**2. 为什么会有“一圈圈波纹”？**

振铃的根源在于：**频域的“硬边界” → 空间域的“振荡”**。

**关键定理（傅里叶变换对偶性）**：

| 频域 | 空间域 |
|------|--------|
| 理想低通的传递函数 = **矩形窗**（硬边界，从 1 突变为 0） | 矩形窗的傅里叶逆变换 = **sinc 函数** |

$$
\text{sinc}(x) = \frac{\sin(\pi x)}{\pi x}
$$

- sinc 函数的形状：**中心一个主瓣，两侧有无限多个逐渐衰减的旁瓣**；
- 空间域卷积一个 sinc 函数，就相当于在每个像素周围加上一圈圈**正负交替的波纹**；
- 这就是我们在图像边缘看到的“一圈圈波纹”——**振铃效应（Ringing）**。

**一句话概括**：

> **频域的“一刀切”，在空间域就是“一圈圈的波纹”。**

**3. 直观类比：为什么“硬切”会产生波纹？**

想象你在听一首音乐，突然用剪刀把高频部分“一刀剪掉”：

- **剪得平滑**（渐变过渡，像高斯那样）→ 声音听起来自然；
- **剪得突然**（硬边界，像理想低通那样）→ 会听到“咔哒”声或“嗡嗡”声——这就是**时域的振铃**。

图像里的振铃同理：

| 频域操作 | 空间域表现 |
|---------|-----------|
| 硬边界（突变） | 信号被“突然切断” → 产生振荡 → 一圈圈波纹 |
| 平滑过渡 | 信号被“逐渐衰减” → 无振荡 → 无振铃 |

**4. 为什么巴特沃斯和高斯没有振铃？**

| 滤波器 | 频域过渡带 | 空间域核 | 振铃 |
|--------|-----------|---------|------|
| 理想低通 | 无（突变） | sinc（无限振荡） | **明显** |
| 巴特沃斯低通 | 较宽（平滑过渡） | 介于 sinc 和高斯之间 | 较小 |
| 高斯低通 | 最宽（高斯衰减） | 高斯核（无振荡） | **几乎无** |

**关键规律**：

> **频域过渡越平滑，空间域振铃越小。**
>
> - 理想低通：过渡带宽度 = 0（硬边界）→ 振铃最严重；
> - 巴特沃斯：过渡带宽度可调 → 振铃可控；
> - 高斯：过渡带最宽（平滑衰减）→ 几乎无振铃。

**5. 与空间域滤波的对应**

| 空间域核 | 频域传递函数 | 振铃 |
|---------|------------|------|
| 无限宽的 sinc | 理想低通（矩形） | 明显 |
| 有限宽的高斯核 | 高斯低通 | 几乎无 |
| 有限宽的均值核 | 近似低通 | 较小 |

- 空间域的 **GaussianBlur**（6.4.2 节）对应的正是频域的**高斯低通**，两者都**无振铃**；
- 而空间域的“理想低通”需要无限大的核，实际无法实现。

---

五、结论与选择建议

**1. 三种低通滤波器的对比**

| 滤波器 | 振铃 | 过渡带 | 实际使用 |
|--------|------|--------|---------|
| 理想低通 | 明显 | 无 | 很少用（伪影太重） |
| 巴特沃斯低通 | 较小 | 可调 | 需要精细控制时用 |
| 高斯低通 | 几乎无 | 宽 | 最常用（最自然） |

**2. 实际选择建议**

- **需要最平滑、最自然的结果** → **高斯低通**；
- **需要可调过渡带、精细控制** → **巴特沃斯低通**（通过阶数 n 调节）；
- **只是理论分析、推导公式** → 理想低通。

**3. 一句话总结**

> **理想低通的“理想”是指数学定义上的完美矩形，不是视觉效果上的理想。**
>
> - 它的传递函数是**硬边界**（从 1 突变到 0）；
> - 硬边界在空间域对应 **sinc 函数**，sinc 有无限多个旁瓣；
> - 这些旁瓣叠加到图像上，就是**一圈圈波纹（振铃）**；
> - 过渡带越平滑（巴特沃斯 → 高斯），振铃越小；
> - 所以，实际应用中应选择**平滑过渡**的滤波器，而非“理想”的硬边界。

**记忆口诀**：

> **频域一刀切，空间起波纹；**
> **过渡越平滑，振铃越轻微。**
> **理想不理想，工程看高斯。**

## 7.4 高通滤波：理想、巴特沃斯、高斯高通

### 内容介绍

**高通滤波（High-Pass Filter, HPF）**：抑制低频，保留高频。

**效果**：边缘增强、锐化、提取轮廓。

**三种常见高通滤波器**：

| 滤波器 | 传递函数 | 特点 |
|--------|---------|------|
| **理想高通** | $H(u,v) = \begin{cases} 0, & D(u,v) \le D_0 \\ 1, & D(u,v) > D_0 \end{cases}$ | 硬边界，振铃明显 |
| **巴特沃斯高通** | $H(u,v) = \frac{1}{1 + [D_0/D(u,v)]^{2n}}$ | 软边界，振铃较小 |
| **高斯高通** | $H(u,v) = 1 - e^{-D^2(u,v)/(2D_0^2)}$ | 最平滑，几乎无振铃 |

**与低通的关系**：

$$H_{HP}(u, v) = 1 - H_{LP}(u, v)$$

即：高通 = 1 - 低通。

---

### 一、高通滤波的效果

**视觉效果**：

- 边缘被突出，背景变暗；
- 平坦区域（低频）被抑制；
- 细节、边缘、噪声（高频）被保留。

**典型用途**：

- 边缘检测；
- 图像锐化；
- 纹理分析；
- 去光照不均（低频光照被抑制）。

**截止频率 $D_0$ 的影响**：

| $D_0$ | 效果 |
|-------|------|
| 很小 | 保留大部分频率，接近原图 |
| 中等 | 边缘突出，背景变暗 |
| 很大 | 只保留最强边缘，大部分变黑 |

---

### 二、三种高通滤波器对比

**1. 理想高通（Ideal HPF）**

- 传递函数：硬边界，$D_0$ 内全 0，外全 1；
- 空间域对应：sinc 函数；
- 缺点：**振铃效应**明显；
- 实际中很少直接用。

**2. 巴特沃斯高通（Butterworth HPF）**

- 传递函数：平滑过渡，$D = D_0$ 时 $H = 0.5$；
- 阶数 $n$ 控制过渡带宽度；
- 优点：振铃比理想高通小；
- 常用 $n = 2$ 或 $n = 3$。

**3. 高斯高通（Gaussian HPF）**

- 传递函数：最平滑；
- 空间域对应：高斯核；
- 优点：**几乎无振铃**，过渡最自然；
- 实际中最常用。

**三者对比**：

| 滤波器 | 过渡带 | 振铃 | 空间域核 |
|--------|--------|------|---------|
| 理想 | 无（突变） | 明显 | sinc（无限宽） |
| 巴特沃斯 | 可调 | 较小 | 介于两者之间 |
| 高斯 | 宽 | 几乎无 | 高斯（有限宽） |

---

### 三、高通滤波的传递函数

**理想高通**：

$$H(u, v) = \begin{cases} 0, & D(u, v) \le D_0 \\ 1, & D(u, v) > D_0 \end{cases}$$

**巴特沃斯高通**：

$$H(u, v) = \frac{1}{1 + [D_0/D(u, v)]^{2n}}$$

**高斯高通**：

$$H(u, v) = 1 - e^{-D^2(u, v)/(2D_0^2)}$$

其中：

$$D(u, v) = \sqrt{(u - M/2)^2 + (v - N/2)^2}$$

**关键**：这些传递函数同样是径向对称的，正负频率被同等对待。

---

### 四、代码实现模板（向量化）

```python
import numpy as np
import cv2

def make_distance_grid(shape):
    rows, cols = shape
    crow, ccol = rows // 2, cols // 2
    u, v = np.meshgrid(np.arange(rows), np.arange(cols), indexing='ij')
    return np.sqrt((u - crow)**2 + (v - ccol)**2)

def ideal_highpass(shape, D0):
    D = make_distance_grid(shape)
    return (D > D0).astype(np.float32)

def butterworth_highpass(shape, D0, n=2):
    D = make_distance_grid(shape)
    return 1 / (1 + (D0 / (D + 1e-6))**(2 * n))

def gaussian_highpass(shape, D0):
    D = make_distance_grid(shape)
    return 1 - np.exp(-(D**2) / (2 * D0**2))
```

---

### 五、高通滤波与锐化

**高通滤波 = 提取高频细节**。

**锐化**：

$$I_{sharp} = I + k \cdot I_{highpass}$$

- $I$：原图；
- $I_{highpass}$：高通滤波结果（边缘、细节）；
- $k$：锐化强度；
- 结果：边缘增强，图像更清晰。

**与空间域锐化的对应**：

| 频域 | 空间域 |
|------|--------|
| 高斯高通 | USM 锐化（6.4.6 节） |
| Laplacian 高通 | Laplacian 锐化（6.4.7 节） |

---

### 六、一句话总结

> **高通滤波 = 抑制低频，保留高频 = 边缘增强、锐化。**
>
> - **理想高通**：硬边界，振铃明显；
> - **巴特沃斯高通**：软边界，阶数可调，振铃较小；
> - **高斯高通**：最平滑，几乎无振铃，最常用；
> - **截止频率 $D_0$**：越大保留越少，边缘越突出；越小保留越多，越接近原图。
>
> 高通滤波 + 原图 = 锐化，对应空间域的 USM 和 Laplacian 锐化。

### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

# ============================================================
# 合成一张包含多种频率成分的测试图
# ============================================================
def make_test_image(size=256):
    """合成测试图：灰底 + 白色方块 + 黑色圆 + 细线"""
    img = np.full((size, size), 100, dtype=np.uint8)         # 灰底
    cv2.rectangle(img, (30, 30), (100, 100), 255, -1)        # 白色方块（硬边缘）
    cv2.circle(img, (180, 70), 35, 0, -1)                    # 黑色圆（曲线边缘）
    cv2.line(img, (30, 180), (220, 180), 255, 1)             # 细白线（高频）
    cv2.line(img, (30, 200), (220, 200), 255, 1)             # 细白线（高频）
    return img

def make_distance_grid(shape):                               # 距离矩阵（向量化）
    rows, cols = shape
    crow, ccol = rows // 2, cols // 2
    u, v = np.meshgrid(np.arange(rows), np.arange(cols), indexing='ij')
    return np.sqrt((u - crow)**2 + (v - ccol)**2)

def ideal_highpass(shape, D0):                               # 理想高通
    D = make_distance_grid(shape)
    return (D > D0).astype(np.float32)

def butterworth_highpass(shape, D0, n=2):                    # 巴特沃斯高通
    D = make_distance_grid(shape)
    return 1 / (1 + (D0 / (D + 1e-6))**(2 * n))

def gaussian_highpass(shape, D0):                            # 高斯高通
    D = make_distance_grid(shape)
    return 1 - np.exp(-(D**2) / (2 * D0**2))

img_gray = make_test_image(256)                              # 合成测试图
F = np.fft.fft2(img_gray)                                    # 傅里叶变换
F_shift = np.fft.fftshift(F)                                 # 中心化

D0 = 30                                                      # 截止频率
H_ideal = ideal_highpass(img_gray.shape, D0)                 # 理想高通 H
H_butter = butterworth_highpass(img_gray.shape, D0, n=2)     # 巴特沃斯高通 H
H_gauss = gaussian_highpass(img_gray.shape, D0)              # 高斯高通 H

def apply_filter(F_shift, H):                                # 应用滤波器
    G = F_shift * H                                          # 频域滤波
    G_unshift = np.fft.ifftshift(G)                          # 反中心化
    return np.abs(np.fft.ifft2(G_unshift))                   # 逆变换

img_ideal = apply_filter(F_shift, H_ideal)                   # 理想高通结果
img_butter = apply_filter(F_shift, H_butter)                 # 巴特沃斯高通结果
img_gauss = apply_filter(F_shift, H_gauss)                   # 高斯高通结果

# ---------- 可视化 ----------
plt.figure(figsize=(16, 10))                                 # 创建画布

plt.subplot(2, 4, 1)                                         # 子图 1
plt.title('合成测试图\n(灰底+方块+圆+细线)')                    # 标题
plt.imshow(img_gray, cmap='gray', vmin=0, vmax=255)          # 显示原图
plt.axis('off')                                              # 关闭坐标轴

plt.subplot(2, 4, 2)                                         # 子图 2
plt.title('理想高通 H')                                       # 标题
plt.imshow(H_ideal, cmap='gray', vmin=0, vmax=1)             # 显示 H
plt.axis('off')                                              # 关闭坐标轴

plt.subplot(2, 4, 3)                                         # 子图 3
plt.title('巴特沃斯高通 H')                                   # 标题
plt.imshow(H_butter, cmap='gray', vmin=0, vmax=1)            # 显示 H
plt.axis('off')                                              # 关闭坐标轴

plt.subplot(2, 4, 4)                                         # 子图 4
plt.title('高斯高通 H')                                       # 标题
plt.imshow(H_gauss, cmap='gray', vmin=0, vmax=1)             # 显示 H
plt.axis('off')                                              # 关闭坐标轴

plt.subplot(2, 4, 5)                                         # 子图 5
plt.title('理想高通结果\n(振铃明显)')                          # 标题
plt.imshow(img_ideal, cmap='gray', vmin=0, vmax=255)         # 显示结果
plt.axis('off')                                              # 关闭坐标轴

plt.subplot(2, 4, 6)                                         # 子图 6
plt.title('巴特沃斯高通结果\n(振铃较小)')                      # 标题
plt.imshow(img_butter, cmap='gray', vmin=0, vmax=255)        # 显示结果
plt.axis('off')                                              # 关闭坐标轴

plt.subplot(2, 4, 7)                                         # 子图 7
plt.title('高斯高通结果\n(几乎无振铃)')                        # 标题
plt.imshow(img_gauss, cmap='gray', vmin=0, vmax=255)         # 显示结果
plt.axis('off')                                              # 关闭坐标轴

plt.tight_layout()                                           # 自动调整间距
plt.show()                                                   # 显示图像

**图像解读**

一、七张子图分别是什么

| 子图 | 内容 | 对应代码 |
|------|------|---------|
| 第 1 行第 1 列 | 合成测试图 | `img_gray` |
| 第 1 行第 2 列 | 理想高通 H | `H_ideal` |
| 第 1 行第 3 列 | 巴特沃斯高通 H | `H_butter` |
| 第 1 行第 4 列 | 高斯高通 H | `H_gauss` |
| 第 2 行第 1 列 | 理想高通结果 | `img_ideal` |
| 第 2 行第 2 列 | 巴特沃斯高通结果 | `img_butter` |
| 第 2 行第 3 列 | 高斯高通结果 | `img_gauss` |

---

二、观察到的现象

**三种传递函数 H**

| H | 特点 |
|---|------|
| 理想高通 | 中心黑色圆，边界锐利，外围纯白 |
| 巴特沃斯高通 | 中心黑色，边缘平滑过渡，外围灰色渐变 |
| 高斯高通 | 中心黑色，过渡最平滑，外围灰色渐变 |

**三种滤波结果**

| 结果 | 特点 |
|------|------|
| 理想高通 | 背景大面积变黑，边缘清晰，有**振铃** |
| 巴特沃斯高通 | 背景变暗，边缘清晰，振铃比理想高通**小** |
| 高斯高通 | 背景变暗，边缘清晰，**几乎无振铃**，最自然 |

---

三、这说明了什么

**1. 高通滤波抑制低频**

- 低频（整体亮度、大结构）被抑制 → 背景变暗；
- 高频（边缘、细节）被保留 → 边缘突出；
- 结果：像一张"描边图"。

**2. 振铃程度不同**

- 理想高通：硬边界 → 振铃明显；
- 巴特沃斯：软边界 → 振铃较小；
- 高斯：最平滑 → 几乎无振铃。

**3. 高通滤波和低通滤波的对偶**

$$H_{HP} = 1 - H_{LP}$$

- 低通：保留中心，抑制外围；
- 高通：抑制中心，保留外围；
- 两者互补，相加 = 1。

**4. 高通滤波与锐化**

- 高通结果 = 边缘和细节；
- 原图 + k × 高通结果 = 锐化图像；
- 对应空间域的 USM 锐化（6.4.6 节）和 Laplacian 锐化（6.4.7 节）。

**5. 实际选择建议**

- **高斯高通**：最常用，几乎无振铃；
- **巴特沃斯高通**：需要可调过渡带时用；
- **理想高通**：理论分析用，实际很少用。

---

四、小结

> 这张图用合成测试图对比了三种高通滤波器：
>
> - **理想高通**：硬边界，振铃明显；
> - **巴特沃斯高通**：软边界，振铃较小；
> - **高斯高通**：最平滑，几乎无振铃，最常用；
> - 三种都突出边缘、抑制背景，但振铃程度不同；
> - 高通 + 原图 = 锐化，对应空间域的 USM 和 Laplacian 锐化。

## 7.5  Laplacian 算子

### 内容介绍

**Laplacian 算子**在频域中有明确的传递函数，是**高通滤波的一种特殊形式**。

**空间域 Laplacian**（6.4.7 节）：

$$\nabla^2 I = \frac{\partial^2 I}{\partial x^2} + \frac{\partial^2 I}{\partial y^2}$$

**频域 Laplacian 传递函数**：

$$H(u, v) = -4\pi^2 (u^2 + v^2)$$

或者写成：

$$H(u, v) = -(u^2 + v^2)$$

（常数因子可忽略，因为最终会归一化）

**关键点**：

- Laplacian 的传递函数与 $D^2$ 成正比；
- $D$ 越大（频率越高），$H$ 越大；
- 所以 Laplacian 是**高通滤波**；
- 但它和理想/巴特沃斯/高斯高通不同：
  - 上述三种是"先低频全抑制，再逐渐通过高频"；
  - Laplacian 是"频率越高，通过得越多"，没有明确的截止频率。

---

### 一、频域 Laplacian 的传递函数

**连续域**：

$$H(u, v) = -4\pi^2 (u^2 + v^2)$$

**离散域**（以中心化频谱为例）：

$$H(u, v) = -(u^2 + v^2)$$

其中 $u, v$ 是相对于频谱中心的坐标：

$$u = x - M/2, \quad v = y - N/2$$

**注意**：$H$ 关于 0 Hz 对称，正负频率同样加权，符合共轭对称要求。

---

### 二、频域 Laplacian 与空间域 Laplacian 的对应

| 空间域 | 频域 |
|--------|------|
| 4 邻域核 `[[0,1,0],[1,-4,1],[0,1,0]]` | $H = -(u^2 + v^2)$ |
| 8 邻域核 `[[1,1,1],[1,-8,1],[1,1,1]]` | $H = -(u^2 + v^2)$ |
| 二阶偏导 | 频率平方加权 |

**关键**：

- 空间域的 Laplacian 核（差分）↔ 频域的 $D^2$ 加权；
- 两者数学等价；
- 空间域实现简单，频域实现直观。

---

### 三、梯度算子的传递函数（重要补充）

**1. 梯度算子的定义**

梯度算子（如 Sobel、Prewitt、Roberts）在空间域是一阶差分：

$$G_x = \frac{\partial I}{\partial x}, \quad G_y = \frac{\partial I}{\partial y}$$

**2. 一阶微分的频域传递函数**

由傅里叶变换的微分性质：

$$\mathcal{F}\left\{\frac{\partial I}{\partial x}\right\} = j 2\pi u \cdot F(u, v)$$

$$\mathcal{F}\left\{\frac{\partial I}{\partial y}\right\} = j 2\pi v \cdot F(u, v)$$

所以：

- **x 方向梯度**的传递函数：$H_x(u, v) = j 2\pi u$
- **y 方向梯度**的传递函数：$H_y(u, v) = j 2\pi v$

**3. 梯度幅值的传递函数**

梯度幅值：

$$|\nabla I| = \sqrt{G_x^2 + G_y^2}$$

在频域中，它的传递函数不是简单的乘积，但可以近似为：

$$H_{grad}(u, v) = \sqrt{u^2 + v^2} = D(u, v)$$

**4. 梯度算子 vs Laplacian**

| 算子 | 阶数 | 频域传递函数 | 对频率的加权 |
|------|------|------------|------------|
| 梯度（Sobel 等） | 一阶 | $D$ | 线性加权 |
| Laplacian | 二阶 | $D^2$ | 平方加权 |

**5. 直观理解**

- 梯度算子：频率越高，通过越多（线性）；
- Laplacian：频率越高，通过越多（平方），对高频更敏感；
- 所以 Laplacian 对噪声比梯度算子更敏感。

**6. 代码示例**

```python
def gradient_freq(shape):
    """梯度算子的频域传递函数 H = D"""
    D = make_distance_grid(shape)
    return D / np.max(D)  # 归一化
```

---

### 四、频域 Laplacian 锐化

**锐化公式**（与空间域一致）：

$$I_{sharp} = I - k \cdot \nabla^2 I$$

**频域实现**：

1. 傅里叶变换：$F = FFT(I)$；
2. 中心化：$F_{shift} = fftshift(F)$；
3. 构造 Laplacian 传递函数：$H = -(u^2 + v^2)$；
4. 频域滤波：$G = F_{shift} \cdot (1 + k \cdot H)$；
   - 注意：这里不是 $F_{shift} \cdot H$，而是 $F_{shift} \cdot (1 + kH)$；
   - 因为锐化 = 原图 + k × Laplacian；
   - 频域对应：$1 + k \cdot H$；
5. 反中心化：$G_{unshift} = ifftshift(G)$；
6. 逆变换：$I_{sharp} = IFFT(G_{unshift})$。

**关键**：

- 纯 Laplacian（$H$）提取高频；
- 锐化（$1 + kH$）保留原图 + 增强高频；
- 如果只做 $H$，得到的是边缘图，不是锐化图。

---

### 五、代码实现模板

```python
import numpy as np
import cv2

def laplacian_freq(shape):
    rows, cols = shape
    crow, ccol = rows // 2, cols // 2
    H = np.zeros((rows, cols), np.float32)
    for u in range(rows):
        for v in range(cols):
            du = u - crow
            dv = v - ccol
            H[u, v] = -(du**2 + dv**2)
    H = H / np.max(np.abs(H))   # 归一化到 [-1, 1]
    return H

img = cv2.imread('image.jpg', 0)
F = np.fft.fft2(img)
F_shift = np.fft.fftshift(F)

H = laplacian_freq(img.shape)
k = 0.5
G = F_shift * (1 + k * H)
G_unshift = np.fft.ifftshift(G)
img_sharp = np.abs(np.fft.ifft2(G_unshift))
```

---

### 六、一句话总结

> **频域 Laplacian = 传递函数 $H(u,v) = -(u^2 + v^2)$ 的高通滤波。**
>
> - 频率越高，通过越多；
> - 没有截止频率，所有高频按 $D^2$ 加权；
> - 与空间域 Laplacian 核数学等价；
> - 梯度算子传递函数为 $H = D$（一阶），Laplacian 为 $H = D^2$（二阶）；
> - 锐化公式：$I_{sharp} = I - k\nabla^2 I$，频域对应 $1 + kH$；
> - 纯 $H$ 提取边缘，$1 + kH$ 实现锐化。

### 示例

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ============================================================
# 1. 合成一张典型测试图
# ============================================================
def make_test_image(size=256):
    """合成测试图：灰底 + 白色方块 + 黑色圆 + 细线 + 噪点"""
    img = np.full((size, size), 128, dtype=np.uint8)         # 灰底 128
    cv2.rectangle(img, (30, 30), (100, 100), 255, -1)        # 白色方块（硬边缘）
    cv2.circle(img, (180, 70), 35, 0, -1)                    # 黑色圆（曲线边缘）
    cv2.line(img, (30, 180), (220, 180), 255, 1)             # 细白线（高频）
    cv2.line(img, (30, 200), (220, 200), 255, 1)             # 细白线（高频）
    # 添加随机噪点（高频）
    noise = np.random.randint(0, 50, (size, size), dtype=np.uint8)
    img = np.clip(img.astype(np.int32) + noise - 25, 0, 255).astype(np.uint8)
    return img

# ============================================================
# 2. 频域 Laplacian 传递函数（向量化）
# ============================================================
def laplacian_freq(shape):
    """频域 Laplacian 传递函数：H = -(u^2 + v^2)，归一化到 [0, 1]"""
    rows, cols = shape
    crow, ccol = rows // 2, cols // 2
    u, v = np.meshgrid(np.arange(rows), np.arange(cols), indexing='ij')
    H = -((u - crow)**2 + (v - ccol)**2)
    H = np.abs(H)
    H = H / np.max(H)        # 归一化到 [0, 1]
    return H

# ============================================================
# 3. 合成图 → 频域滤波
# ============================================================
img_gray = make_test_image(256)                              # 合成测试图
F = np.fft.fft2(img_gray)                                    # 傅里叶变换
F_shift = np.fft.fftshift(F)                                 # 中心化

H = laplacian_freq(img_gray.shape)                           # Laplacian 传递函数

# 纯 Laplacian（边缘提取）
G_lap = F_shift * H                                          # 频域滤波
G_lap_unshift = np.fft.ifftshift(G_lap)                      # 反中心化
img_lap_raw = np.fft.ifft2(G_lap_unshift)                    # 逆变换
img_lap = np.abs(img_lap_raw)                                # 取模长
img_lap = cv2.normalize(img_lap, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

# ============================================================
# 4. 可视化
# ============================================================
plt.figure(figsize=(12, 5))

plt.subplot(1, 3, 1)
plt.title('合成测试图\n(灰底+方块+圆+细线+噪点)')
plt.imshow(img_gray, cmap='gray', vmin=0, vmax=255)
plt.axis('off')

plt.subplot(1, 3, 2)
plt.title('Laplacian 传递函数 H\n(中心 0，向外平方增长)')
plt.imshow(H, cmap='gray', vmin=0, vmax=1)
plt.axis('off')

plt.subplot(1, 3, 3)
plt.title('纯 Laplacian（边缘提取）')
plt.imshow(img_lap, cmap='gray', vmin=0, vmax=255)
plt.axis('off')

plt.tight_layout()
plt.show()

**图像解读**

一、三张子图分别是什么

| 位置 | 子图 | 说明 |
|------|------|------|
| 左：合成测试图 | 灰底 + 白色方块 + 黑色圆 + 细线 + 噪点 | 输入图像 |
| 中：Laplacian 传递函数 H | 中心 0，向外平方增长 | 频域 Laplacian 滤波器 |
| 右：纯 Laplacian（边缘提取） | 平坦区黑，边缘和细线被提取 | Laplacian 的输出 |

---

二、Laplacian 传递函数 H：中心 0，向外平方增长

**1. 传递函数的形态**

- 中心黑色（H = 0）→ 完全抑制低频；
- 向外逐渐变亮（H 增大）→ 高频通过得更多；
- 四个角最亮（H 最大）→ 最高频完全通过。

**2. 公式**

$$
H(u, v) = -(u^2 + v^2)
$$

其中 u、v 是相对于频谱中心的坐标，归一化后显示为 [0, 1]。

**3. 关键特征**

| 特征 | 说明 |
|------|------|
| 中心为 0 | 完全抑制低频（平坦区、整体亮度） |
| 向外平方增长 | 频率越高，通过越多 |
| 没有截止频率 | 与理想/巴特沃斯/高斯高通不同，它没有 D0 |
| 与空间域 Laplacian 等价 | 对应核 `[[0,1,0],[1,-4,1],[0,1,0]]` |
| 径向对称 | 正负频率同等加权，符合共轭对称 |

---

三、纯 Laplacian（边缘提取）：输出是一张边缘图

**1. 平坦区（灰底）**

- 灰底是低频，Laplacian 完全抑制 → 输出接近黑色。

**2. 边缘（方块和圆）**

- 方块和圆的边缘是高频，Laplacian 响应强烈 → 输出为亮线。
- 因为 Laplacian 是二阶导数，边缘两侧符号相反，取模长后表现为一条亮线。

**3. 细线**

- 细白线属于极高频，Laplacian 对其非常敏感 → 输出为清晰的亮线。

**4. 噪点**

- 随机噪点也是高频，Laplacian 对其同样敏感 → 输出为散乱的白点。
- 这暴露了 Laplacian 的缺点：**对噪声敏感**。

---

四、Laplacian 的用途

| 用途 | 说明 |
|------|------|
| 边缘检测 | 对像素值突变敏感，能检测边缘、角点、细线 |
| 细节提取 | 提取高频信息，如纹理、细小结构 |
| 各向同性 | 4 邻域和 8 邻域核都具有旋转不变性 |

**一句话总结**：

> **频域 Laplacian 的传递函数是 H = -(u² + v²)，中心 0，向外平方增长。**
> **它是典型的高通滤波器，用来提取边缘、细线等高频信息。**
> **平坦区被完全抑制（变黑），边缘和细节被完整保留。**

## 7.6 选择性滤波器：带阻滤波器、带通滤波器

### 内容介绍

**选择性滤波器**：只保留或抑制**某个频率范围**的滤波器。

| 滤波器 | 作用 | 传递函数（以巴特沃斯为例） |
|--------|------|--------------------------|
| **带阻滤波器（Band-Reject）** | 抑制某个频率范围，保留其他 | $H = \frac{1}{1 + [D_0 W / (D^2 - D_0^2)]^{2n}}$ |
| **带通滤波器（Band-Pass）** | 保留某个频率范围，抑制其他 | $H = 1 - H_{bandreject}$ |

- $D_0$：中心频率；
- $W$：带宽；
- $n$：阶数。

**典型用途**：

- **带阻**：去除周期性噪声（条纹、网纹）；
- **带通**：提取特定频率的纹理或结构。

**注意**：带阻和带通都是**径向对称**的，同时作用在正负频率上——这是保持共轭对称的必要条件。

---

### 一、为什么需要带阻/带通

**低通**：保留低频，抑制高频；
**高通**：保留高频，抑制低频；
**带阻**：抑制中间某个频段，保留低频和高频；
**带通**：保留中间某个频段，抑制低频和高频。

**典型场景**：

- 图像上有周期性条纹（比如扫描线、网纹）；
- 条纹在频谱上表现为**一对或几对亮点**；
- 用带阻滤波器把这些亮点抹掉，条纹就消失了；
- 这就是**陷波滤波器**（7.7 节）的思想。

---

### 二、带阻滤波器的传递函数

**巴特沃斯带阻**：

$$H(u, v) = \frac{1}{1 + \left[\frac{D(u,v) W}{D^2(u,v) - D_0^2}\right]^{2n}}$$

- $D_0$：要抑制的中心频率；
- $W$：抑制带宽；
- $n$：阶数，越大过渡越陡；
- $D(u,v) = D_0$ 时，$H = 0$（完全抑制）；
- $D$ 远离 $D_0$ 时，$H \to 1$（通过）。

**高斯带阻**：

$$H(u, v) = 1 - e^{-\frac{1}{2}\left[\frac{D^2(u,v) - D_0^2}{D(u,v) W}\right]^2}$$

**图形**：

```text
H(u,v)
  1 │──────       ──────
  0 │      ───────
    └──────────────────→ D
              D0
```

- $D_0$ 附近：$H = 0$（抑制）；
- 远离 $D_0$：$H = 1$（通过）。

---

### 三、带通滤波器的传递函数

**带通 = 1 - 带阻**：

$$H_{BP}(u, v) = 1 - H_{BR}(u, v)$$

**图形**：

```text
H(u,v)
  1 │      ───────
  0 │──────       ──────
    └──────────────────→ D
             D0
```

- $D_0$ 附近：$H = 1$（通过）；
- 远离 $D_0$：$H = 0$（抑制）。

---

### 四、代码实现模板

```python
import numpy as np
import cv2

def butterworth_bandreject(shape, D0, W, n=2):
    rows, cols = shape
    crow, ccol = rows // 2, cols // 2
    H = np.zeros((rows, cols), np.float32)
    for u in range(rows):
        for v in range(cols):
            D = np.sqrt((u - crow)**2 + (v - ccol)**2)
            if abs(D - D0) < 1e-6:
                H[u, v] = 0
            else:
                H[u, v] = 1 / (1 + (D * W / (D**2 - D0**2))**(2 * n))
    return H

def butterworth_bandpass(shape, D0, W, n=2):
    return 1 - butterworth_bandreject(shape, D0, W, n)
```

---

### 五、带阻和带通的对偶关系

$$H_{BP} = 1 - H_{BR}$$

- 带阻抑制的，正是带通保留的；
- 带阻保留的，正是带通抑制的；
- 两者互补，相加 = 1。

**类比**：

| 滤波器 | 类比 |
|--------|------|
| 低通 | 只让低音通过 |
| 高通 | 只让高音通过 |
| 带阻 | 掐掉中音，保留高音和低音 |
| 带通 | 只让中音通过，掐掉高音和低音 |

---

### 六、一句话总结

> **选择性滤波器 = 只保留/抑制某个频率范围。**
>
> - **带阻**：抑制中间频段，保留低频和高频；
> - **带通**：保留中间频段，抑制低频和高频；
> - **带通 = 1 - 带阻**；
> - 典型用途：去除周期性噪声（带阻）、提取特定纹理（带通）。

### 示例

In [ ]:
import cv2                                                      # 导入 OpenCV
import numpy as np                                              # 导入 NumPy
import matplotlib.pyplot as plt                                 # 导入 Matplotlib

plt.rcParams['font.sans-serif'] = ['SimHei']                    # 中文显示
plt.rcParams['axes.unicode_minus'] = False                      # 负号正常显示

# ============================================================
# 1. 合成一张带周期性条纹的测试图
# ============================================================
def make_test_image_with_stripes(size=256):                     # 定义合成函数
    """合成测试图：灰底 + 方块 + 圆 + 水平正弦条纹"""            # 函数说明
    img = np.full((size, size), 100, dtype=np.uint8)            # 灰底 100
    cv2.rectangle(img, (30, 30), (100, 100), 255, -1)           # 白色方块
    cv2.circle(img, (180, 70), 35, 0, -1)                       # 黑色圆
    for i in range(size):                                       # 逐行遍历
        img[i, :] = np.clip(img[i, :].astype(np.int32) +        # 叠加正弦条纹
                            40 * np.sin(2 * np.pi * i / 20),    # 周期 20 像素
                            0, 255).astype(np.uint8)            # 截断到 0~255
    return img                                                  # 返回合成图

def butterworth_bandreject(shape, D0, W, n=2):                  # 巴特沃斯带阻
    rows, cols = shape                                          # 图像尺寸
    crow, ccol = rows // 2, cols // 2                           # 频谱中心
    H = np.zeros((rows, cols), np.float32)                      # 初始化 H
    for u in range(rows):                                       # 遍历行
        for v in range(cols):                                   # 遍历列
            D = np.sqrt((u - crow)**2 + (v - ccol)**2)          # 到中心距离
            if abs(D - D0) < 1e-6:                              # 避免除零
                H[u, v] = 0                                     # D0 处设 0
            else:                                               # 一般情况
                H[u, v] = 1 / (1 + (D * W /                      # 巴特沃斯公式
                             (D**2 - D0**2))**(2 * n))          # 带阻传递函数
    return H                                                    # 返回 H

img_gray = make_test_image_with_stripes(256)                    # 合成测试图
F = np.fft.fft2(img_gray)                                       # 傅里叶变换
F_shift = np.fft.fftshift(F)                                    # 频谱中心化

D0 = 13                                                         # 中心频率
W = 8                                                           # 带宽
H_br = butterworth_bandreject(img_gray.shape, D0, W)            # 带阻 H
H_bp = 1 - H_br                                                 # 带通 H

# ---------- 带阻滤波 ----------
G_br = F_shift * H_br                                           # 频域乘法
G_br_unshift = np.fft.ifftshift(G_br)                           # 反中心化
img_br = np.abs(np.fft.ifft2(G_br_unshift))                     # 逆变换
img_br = cv2.normalize(img_br, None, 0, 255,                    # 归一化到 0~255
                       cv2.NORM_MINMAX).astype(np.uint8)        # 转 uint8

# ---------- 带通滤波 ----------
G_bp = F_shift * H_bp                                           # 频域乘法
G_bp_unshift = np.fft.ifftshift(G_bp)                           # 反中心化
img_bp = np.abs(np.fft.ifft2(G_bp_unshift))                     # 逆变换
img_bp = cv2.normalize(img_bp, None, 0, 255,                    # 归一化放大弱信号
                       cv2.NORM_MINMAX).astype(np.uint8)        # 转 uint8

# ---------- 可视化 ----------
plt.figure(figsize=(15, 10))                                    # 创建画布

plt.subplot(2, 3, 1)                                            # 第一行第 1 列
plt.title('合成测试图\n(灰底+方块+圆+正弦条纹)')                  # 标题
plt.imshow(img_gray, cmap='gray', vmin=0, vmax=255)             # 显示原图
plt.axis('off')                                                 # 关闭坐标轴

plt.subplot(2, 3, 2)                                            # 第一行第 2 列
plt.title('带阻传递函数 H\n(抑制 D0 附近的环带)')                 # 标题
plt.imshow(H_br, cmap='gray', vmin=0, vmax=1)                   # 显示 H
plt.axis('off')                                                 # 关闭坐标轴

plt.subplot(2, 3, 3)                                            # 第一行第 3 列
plt.title('带通传递函数 H\n(保留 D0 附近的环带)')                 # 标题
plt.imshow(H_bp, cmap='gray', vmin=0, vmax=1)                   # 显示 H
plt.axis('off')                                                 # 关闭坐标轴

plt.subplot(2, 3, 5)                                            # 第二行第 2 列
plt.title('带阻结果\n(条纹被去除)')                              # 标题
plt.imshow(img_br, cmap='gray')                                 # 显示带阻结果
plt.axis('off')                                                 # 关闭坐标轴

plt.subplot(2, 3, 6)                                            # 第二行第 3 列
plt.title('带通结果\n(只保留条纹，已归一化)')                     # 标题
plt.imshow(img_bp, cmap='gray')                                 # 显示带通结果
plt.axis('off')                                                 # 关闭坐标轴

plt.tight_layout()                                              # 自动调整间距
plt.show()                                                      # 显示图像

**图像解读**

一、五张子图分别是什么

| 位置 | 子图 | 对应代码 | 说明 |
|------|------|---------|------|
| 上排左 | 合成测试图 | `img_gray` | 灰底 + 白方块 + 黑圆 + 水平正弦条纹 |
| 上排中 | 带阻传递函数 H | `H_br` | 抑制 D0 附近环带，其余通过 |
| 上排右 | 带通传递函数 H | `H_bp` | 保留 D0 附近环带，其余抑制 |
| 下排中 | 带阻结果 | `img_br` | 条纹被去除，低频结构保留 |
| 下排右 | 带通结果 | `img_bp` | 只保留条纹，已归一化显示 |

> **注意**：下排两张结果图**整体右移一格**（占据第 2、3 列），是为了与上排三张传递函数**列对齐**，视觉上更整齐。

---

二、合成测试图：三种频率成分共存

合成图里**同时包含三种关键频率成分**，是为了让带阻/带通的效果一眼可辨：

| 元素 | 频率归属 | 在带阻/带通中的命运 |
|------|---------|-------------------|
| 灰底（100） | 低频（直流） | 带阻保留、带通抑制 |
| 白色方块 | 中高频（硬边缘） | 带阻保留、带通抑制 |
| 黑色圆 | 中高频（曲边缘） | 带阻保留、带通抑制 |
| **水平正弦条纹**（周期 20） | **中频（≈ 256/20 ≈ 12.8）** | **带阻抑制、带通保留** |

**关键设计**：正弦条纹的基频约 **12.8**，因此取

```python
D0 = 13     # 中心频率，匹配条纹频率
W = 8       # 带宽
```

条纹频率正好落在带阻的"抑制环带"内、带通的"保留环带"内。这样：

- **带阻** → 条纹被掐掉 → 剩灰底、方块、圆；
- **带通** → 条纹被留下 → 灰底、方块、圆被抑制。

---

三、带阻传递函数 H：留外、抑中

观察上排中间的 H 图：

- **中心白色圆**：低频（D < D0 - W/2）→ 通过；
- **中间黑色环带**：D ≈ D0 附近 → 抑制；
- **外围白色**：高频（D > D0 + W/2）→ 通过。

**形状记忆**：像一枚"甜甜圈"——**中心实心白、中间空心黑、外圈实心白**。

巴特沃斯带阻的公式：

$$
H_{BR}(u,v) = \frac{1}{1 + \left[\dfrac{D(u,v)\,W}{D^2(u,v) - D_0^2}\right]^{2n}}
$$

- $D = D_0$ 时，分母的分数项趋于无穷 → $H = 0$；
- $D \to 0$ 或 $D \to \infty$ 时，分数项趋于 0 → $H \to 1$。

---

四、带通传递函数 H：抑外、留中

带通 H 与带阻 **完全互补**：

$$
H_{BP}(u,v) = 1 - H_{BR}(u,v)
$$

对照两张传递函数图：

| 位置 | 带阻 H | 带通 H |
|------|--------|--------|
| 中心（低频） | 白（通过） | 黑（抑制） |
| D0 附近（中频） | 黑（抑制） | 白（通过） |
| 外围（高频） | 白（通过） | 黑（抑制） |

**一句话**：带阻"掐中间"，带通"留中间"，两者相加恒等于 1。

---

五、带阻结果：条纹被去除，低频结构保留

带阻结果（下排中）显示了预期效果：

- **水平条纹几乎完全消失** → 验证"带阻抑制 D0 附近频段"；
- **灰底仍在**（亮度基本不变）→ 验证"低频通过"；
- **方块和圆轮廓保留** → 验证"中高频边缘也通过"；
- **边缘有轻微振铃**（方块和圆外一圈淡晕）→ 巴特沃斯软边界仍有轻微振铃。

这正是**带阻滤波器去除周期性噪声**的典型表现：只掐掉噪声频率，保留图像主体。

---

六、带通结果：只保留条纹，但条纹"变宽变软"了

带通结果（下排右，已归一化）显示了预期效果，但有一个**必须说明的现象**：

**现象 1：只有条纹被保留**

- 灰底、方块、圆都被抑制（背景变暗）；
- 水平条纹清晰可见 → 验证"带通保留 D0 附近频段"。

**现象 2：条纹比原图"变宽、变软、变少"**

仔细对比原图和带通结果，你会发现带通后的条纹**不是原条纹的复制**，而是**变了形状**：

| 对比项 | 原图条纹 | 带通后条纹 |
|--------|---------|-----------|
| 波形 | 近似方波（陡峭） | 近似正弦（平滑） |
| 视觉宽度 | 窄、密集 | 宽、稀疏 |
| 边缘锐度 | 锐利 | 柔和 |

**为什么会这样？（重要）**

**这不是错误，而是带通滤波的固有特性。**

原因分三步：

**第 1 步：原图条纹不是"纯正弦"**

原图叠加的是 `40·sin(2πi/20)`，理论上是一个纯正弦波。但它被叠加到灰底和方块/圆上后经过 `np.clip(..., 0, 255)` 截断，**截断是非线性操作，会引入高次谐波**（3f0、5f0……）。所以原图条纹的频谱不止有两个点，而是有一串谐波。

**第 2 步：带通滤波器保留的是"一整段"频率，不是"一个点"**

```python
D0 = 13, W = 8  →  带通保留 D ∈ [9, 17] 这一段
```

- 条纹基频 f0 ≈ 12.8，落在环带内 → **保留**；
- 条纹谐波 3f0 ≈ 38.4，落在环带外 → **抑制**；
- 环带内的其他频率成分（来自方块/圆边缘的调制） → **也保留**。

**第 3 步：只剩基频附近的条纹，自然"退化"为正弦**

- 原图条纹 = 基频 + 丰富谐波 → 波形陡峭，像方波；
- 带通后条纹 = 基频 + 少量邻近频率，谐波被砍 → 波形平滑，像正弦；
- 视觉上就表现为"条纹变宽、变软、变少"。

**一句话概括**：

> **原图条纹是"方波型"（含谐波，边缘陡）→ 带通后是"正弦型"（只剩基频附近，边缘软）→ 视觉上条纹变宽、变稀。这是"带通只保留一段频率"的必然结果。**

**现象 3：方块和圆的轮廓"浮现"出来**

带通结果里，方块和圆的位置能看到一圈亮边。这**不是带通"保留了边缘"**，而是：

- 边缘的频谱很宽，虽然大部分高频被抑制，但**落在 D ∈ [9, 17] 这一段的中频分量仍然通过**；
- 这些中频分量本身很弱，但**归一化把它们提亮了**，所以视觉上显现出来。

**提示**：标题里已注明"已归一化"，就是提醒读者——**弱信号被放大显示，不代表它们能量强**。

---

七、带阻与带通的对偶关系（本节核心）

$$\boxed{H_{BP} = 1 - H_{BR}}$$

用这张图验证：

| 频率成分 | 带阻处理 | 带通处理 | 相加 |
|---------|---------|---------|------|
| 低频（灰底） | 通过（1） | 抑制（0） | 1 |
| 中频（条纹，D0 附近） | 抑制（0） | 通过（1） | 1 |
| 高频（边缘，远处） | 通过（1） | 抑制（0） | 1 |

**观察两张结果图的互补性**：

- 带阻结果里的东西（灰底、方块、圆），正是带通结果里缺的；
- 带通结果里的东西（条纹），正是带阻结果里缺的；
- 两张结果加起来，等于原图。

**这就是"选择性滤波器"的核心**：**带阻和带通，是同一把"筛子"的正反两面。**

---

八、教学要点：从 7.6 到 7.7 的伏笔

本节演示了带阻和带通，但留下了一个值得思考的问题：

> **如果我只想去掉条纹、保留其他所有东西（包括条纹以外的纹理），带通行不行？**

**答：不行。** 因为带通会：

1. **掐掉低频**（灰底变暗）；
2. **掐掉高频**（边缘变模糊）；
3. **改变条纹本身的波形**（变宽变软）。

**这正是 7.7 陷波滤波器要解决的问题**：

| 滤波器 | 抑制方式 | 对图像的影响 |
|--------|---------|-------------|
| 带阻 | 抑制**一整段**频带 | 该频段内所有信息丢失 |
| **陷波** | 只抑制**几个点** | 只去掉噪声，其他几乎不变 |

**类比**：

- **带阻** = 用圆规画一个环，把环内的东西全扔掉；
- **陷波** = 用针尖扎几个点，只扔掉那几个点。

所以，**去除周期性噪声时，陷波比带阻更精准**——这是 7.7 节的核心动机。

---

九、一句话总结

> **带阻 = "掐中间"：去掉条纹，保留灰底和边缘；**
> **带通 = "留中间"：保留条纹，去掉灰底和边缘；**
> **带通 = 1 − 带阻：两者互补，相加等于原图。**
>
> **注意**：带通结果里的条纹比原图"变宽变软"，是带通只保留一段频率、丢掉谐波的正常结果，不是错误。这个现象恰好说明：
>
> **带通适合"提取某个频段"，但不适合"精准去噪"——精准去噪要用陷波（7.7 节）。**

## 7.7 陷波滤波器

### 内容介绍

**陷波滤波器（Notch Filter）**是**带阻滤波器的特例**：只抑制**特定的几个频率点**（而不是一个连续的频带）。

**典型用途**：去除**周期性噪声**。

**为什么周期性噪声用陷波**：

- 周期性噪声（如扫描线、网纹、摩尔纹）在频谱上表现为**一对或几对亮点**；
- 这些亮点离中心有一定距离，代表特定频率；
- 用陷波滤波器把这些亮点抹掉，噪声就消失了；
- 因为只抹掉几个点，对图像其他频率影响很小。

**为什么陷波必须成对处理（关键依据）**：

实数图像的频谱满足**共轭对称性**：$F(-u,-v) = F^*(u,v)$。也就是说，如果 $(u_k, v_k)$ 处有一个亮点，$(-u_k, -v_k)$ 处必然有一个幅值相同、相位相反的对称点。

- 如果你只抹掉其中一个点，而不抹掉对称点，就**破坏了频谱的共轭对称性**；
- 破坏共轭对称后，逆傅里叶变换的结果**不再是实数图像**（会出现明显的虚部），无法正常显示；
- 所以，陷波滤波必须在**成对的对称位置**同时设 0。

这就是本节反复强调"成对处理"的根本原因。

---

### 一、陷波滤波器的传递函数

**理想陷波**：

$$H(u, v) = \begin{cases} 0, & D_k(u, v) \le D_0 \\ 1, & D_k(u, v) > D_0 \end{cases}$$

**巴特沃斯陷波**：

$$H(u, v) = \prod_{k=1}^{K} \frac{1}{1 + [D_0 / D_k(u, v)]^{2n}}$$

**高斯陷波**：

$$H(u, v) = \prod_{k=1}^{K} \left[1 - e^{-\frac{1}{2}\left(\frac{D_k(u, v)}{D_0}\right)^2}\right]$$

其中：

$$D_k(u, v) = \sqrt{(u - u_k)^2 + (v - v_k)^2}$$

- $(u_k, v_k)$：第 $k$ 个陷波中心；
- $D_0$：陷波半径；
- $K$：陷波个数。

**注意**：

- 频谱是**共轭对称**的；
- 如果 $(u_k, v_k)$ 处有亮点，$(-u_k, -v_k)$ 处也有；
- 所以陷波通常要**成对**处理（中心和对称点都要抹）。

---

### 二、如何找到噪声频率（关键步骤）

陷波滤波最关键、也最容易卡住的一步，就是**找到噪声对应的频率**。实际图像中噪声频率不会写在标题里，必须自己找。常用三种方法。

#### 方法 1：看频谱图上的亮点（最直观）

步骤：

1. 对图像做 FFT + `fftshift` + `log`；
2. 在频谱图上找**偏离中心、成对出现的亮点**；
3. 亮点的位置（距中心的像素数）就是噪声频率。

**关键观察**：

| 亮点位置 | 对应噪声类型 |
|---------|-------------|
| 纵轴上（v 轴） | 水平条纹噪声 |
| 横轴上（u 轴） | 垂直条纹噪声 |
| 斜向 | 斜向条纹噪声 |

**为什么成对？** 因为实信号频谱共轭对称，正频率有一个亮点，负频率对称位置也一定有一个。

#### 方法 2：从空间域估计周期（最快速）

如果能看出条纹的周期 $P$（像素），可直接算频率：

$$
f_{\text{noise}} = \frac{N}{P}
$$

- $N$：图像宽度（或高度）；
- $P$：条纹周期（像素）；
- 例如：图像 256 像素，条纹周期 16 像素 → 频率 ≈ 256/16 = 16。

**直观理解**：

- 条纹越密（周期越小）→ 噪声频率越高 → 亮点离中心越远；
- 条纹越疏（周期越大）→ 噪声频率越低 → 亮点离中心越近。

#### 方法 3：自动检测峰值（最工程化）

在程序里自动找频谱的局部极大值：

1. 把频谱中心附近一小块（直流区域）置零，避免把中心当成峰值；
2. 用 `scipy.signal.find_peaks` 找剩余的局部极大值；
3. 按幅值排序，取最突出的几个；
4. 这些位置就是候选噪声频率。

**注意**：检测到的峰值往往在**正频率一侧**，对应的负频率一侧是对称点，需自己算出。

---

### 三、演示：三种方法找噪声频率

下面用合成图演示三种方法，看看 `noise_freq = 16` 这个数字是怎么来的。

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ============================================================
# 1. 合成一张带周期性条纹噪声的测试图
# ============================================================
def make_noisy_image(size=256):
    """合成测试图：灰底 + 方块 + 圆 + 水平正弦条纹噪声"""
    img = np.full((size, size), 100, dtype=np.uint8)
    cv2.rectangle(img, (30, 30), (100, 100), 255, -1)
    cv2.circle(img, (180, 70), 35, 0, -1)
    # 水平正弦条纹：周期 P = 16 像素
    P = 16
    for i in range(size):
        img[i, :] = np.clip(img[i, :].astype(np.int32) +
                            30 * np.sin(2 * np.pi * i / P), 0, 255).astype(np.uint8)
    return img, P

img_noisy, P_true = make_noisy_image(256)
N = img_noisy.shape[0]

# FFT + 中心化 + 对数
F = np.fft.fft2(img_noisy)
F_shift = np.fft.fftshift(F)
spec = np.log(1 + np.abs(F_shift))

# ============================================================
# 方法 1：看频谱图上的亮点
# ============================================================
plt.figure(figsize=(12, 5))

plt.subplot(1, 3, 1)
plt.title('合成测试图\n(条纹周期 P = 16 像素)')
plt.imshow(img_noisy, cmap='gray', vmin=0, vmax=255)
plt.axis('off')

plt.subplot(1, 3, 2)
plt.title('频谱（中心化 + log）\n纵轴上一对亮点 = 条纹噪声')
plt.imshow(spec, cmap='gray')
# 在频谱上标注亮点位置（视觉辅助）
crow, ccol = N // 2, N // 2
plt.plot([ccol, ccol], [crow - P_true, crow + P_true], 'r+', markersize=12, markeredgewidth=2)
plt.axis('off')

plt.subplot(1, 3, 3)
plt.title('纵轴剖面（v 方向）\n亮点位置 = 噪声频率')
# 取纵轴剖面：v 变化，u = ccol
profile = spec[:, ccol]
plt.plot(profile, 'C0-')
plt.axvline(crow - P_true, color='r', linestyle='--', label=f'亮点: {crow - P_true}')
plt.axvline(crow + P_true, color='r', linestyle='--', label=f'亮点: {crow + P_true}')
plt.xlabel('v（纵轴坐标）')
plt.ylabel('log 幅度')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"图像尺寸 N = {N}")
print(f"条纹真实周期 P = {P_true} 像素")
print(f"对应真实频率 f = N/P = {N}/{P_true} = {N/P_true:.2f}")


**方法 1 解读**：

- **左图**：合成测试图，条纹周期 16 像素。
- **中图**：频谱图。纵轴上、距中心约 16 像素处，有一对红色“+”标记的亮点。
- **右图**：沿纵轴的剖面曲线。曲线在 $v = \text{crow} \pm 16$ 处出现两个峰。
- **结论**：亮点距中心 16 像素 → 噪声频率 = 16。
- **直观规律**：条纹周期 P = 16 → 亮点距中心 = N/P = 256/16 = 16。

> 这正是代码里 `noise_freq = 16` 的来源。

In [ ]:
# ============================================================
# 方法 2：从空间域估计周期
# ============================================================
# 取图像中间一列像素，观察条纹周期
col = img_noisy[:, 10].astype(np.float32)
col = col - col.mean()   # 去掉直流，便于观察波动

plt.figure(figsize=(10, 4))
plt.plot(col, 'C0-')
plt.title('图像第 10 列像素值\n(周期性波动 = 条纹噪声)')
plt.xlabel('行索引 i')
plt.ylabel('像素值（去均值后）')
plt.grid(True)
plt.show()

# 估计周期：找相邻两个波峰的距离
peaks, _ = find_peaks(col)
if len(peaks) >= 2:
    period_est = np.mean(np.diff(peaks))
    print(f"从空间域估计的条纹周期: {period_est:.2f} 像素")
    print(f"对应频率: N / 周期 = {N} / {period_est:.2f} = {N/period_est:.2f}")
else:
    print("未检测到足够波峰")


**方法 2 解读**：

- **图**：图像某一列的像素值，呈现明显的周期性波动。
- **方法**：用 `find_peaks` 找到波峰位置，相邻波峰距离 = 条纹周期。
- **结果**：估计周期约 16 像素 → 频率 ≈ 256/16 = 16。
- **适用场景**：条纹清晰、周期明显时，这种方法最快、最可靠。
- **注意**：如果图像本身有复杂纹理，波动曲线会不干净，这时要配合方法 1 或 3。

In [ ]:
# ============================================================
# 方法 3：自动检测频谱峰值
# ============================================================
# 把中心区域置零，避免把直流当成峰值
spec_no_dc = spec.copy()
R = 5   # 中心屏蔽半径
spec_no_dc[crow - R:crow + R, ccol - R:ccol + R] = 0

# 找局部极大值
peaks_2d, props = find_peaks(spec_no_dc.flatten(), height=np.percentile(spec_no_dc, 99.5))

# 转换为 (u, v) 坐标
peaks_coords = np.array(np.unravel_index(peaks_2d, spec_no_dc.shape)).T

# 按幅值排序，取前 5 个
heights = props['peak_heights']
order = np.argsort(heights)[::-1][:5]
top_peaks = peaks_coords[order]

print("检测到的前 5 个峰值（u, v）及距中心的频率：")
for (u, v) in top_peaks:
    du, dv = u - crow, v - ccol
    freq = np.sqrt(du**2 + dv**2)
    print(f"  (u={u:3d}, v={v:3d})  距中心 = {freq:.2f}")

# 可视化
plt.figure(figsize=(6, 6))
plt.imshow(spec_no_dc, cmap='gray')
plt.title('频谱（中心已屏蔽）\n红圈 = 自动检测到的峰值')
for (u, v) in top_peaks:
    circle = plt.Circle((v, u), 5, color='r', fill=False, linewidth=2)
    plt.gca().add_patch(circle)
plt.axis('off')
plt.show()


**方法 3 解读**：

- **做法**：把频谱中心（直流区域）屏蔽，再找局部峰值。
- **结果**：最突出的两个峰值在纵轴上，距中心约 16 像素。
- **含义**：这两个峰值就是条纹噪声。
- **优点**：自动化，适合批量处理。
- **注意**：
  - 检测到的峰值往往在**正频率一侧**，需自己算出对称的负频率位置；
  - 阈值（这里用 99.5 百分位）需根据图像调整，太低会误检，太高会漏检。

---

**三种方法对比**：

| 方法 | 优点 | 缺点 | 适用场景 |
|------|------|------|---------|
| 1. 看频谱图 | 直观、易懂 | 依赖目视，不够精确 | 教学、单张图 |
| 2. 空间域估计周期 | 最快、最直接 | 需要条纹清晰可辨 | 条纹明显的图 |
| 3. 自动检测峰值 | 自动化、可批量 | 阈值需调参 | 工程、批量处理 |

**实际工作流程**：

1. 先看频谱图（方法 1），确认有亮点；
2. 再看空间域（方法 2），快速估计周期；
3. 最后用自动检测（方法 3），得到精确频率。

三种方法互相验证，确保找对噪声频率。

### 四、陷波滤波的步骤

1. 傅里叶变换：$F = FFT(I)$；
2. 中心化：$F_{shift} = fftshift(F)$；
3. **观察频谱，找到噪声亮点**（方法 1/2/3）；
4. 构造陷波传递函数 $H$：在亮点处设 0，其他设 1；
5. 频域滤波：$G = F_{shift} \cdot H$；
6. 反中心化：$G_{unshift} = ifftshift(G)$；
7. 逆变换：$I_{filtered} = IFFT(G_{unshift})$。

**关键**：第 3 步需要**人工找亮点**，或者用自动检测算法；
**关键**：第 4 步要**成对**设置陷波中心。

### 五、陷波滤波器的应用

| 场景 | 说明 |
|------|------|
| 去除扫描线 | 扫描图像中的水平/垂直条纹 |
| 去除网纹 | 印刷品中的周期性网纹 |
| 去除摩尔纹 | 拍摄屏幕时的干涉条纹 |
| 去除周期噪声 | 传感器或电路引入的周期干扰 |
| 医学图像 | 去除 MRI 中的周期伪影 |

---

### 六、代码实现模板

```python
import numpy as np
import cv2

def notch_filter(shape, centers, D0):
    """
    shape: 图像尺寸
    centers: 陷波中心列表 [(u1,v1), (u2,v2), ...]
    D0: 陷波半径
    """
    rows, cols = shape
    H = np.ones((rows, cols), np.float32)
    for (u_k, v_k) in centers:
        for u in range(rows):
            for v in range(cols):
                D = np.sqrt((u - u_k)**2 + (v - v_k)**2)
                if D <= D0:
                    H[u, v] = 0
    return H

img = cv2.imread('image.jpg', 0)
F = np.fft.fft2(img)
F_shift = np.fft.fftshift(F)

# 先找噪声频率（方法 1/2/3）
noise_freq = 16
rows, cols = img.shape
crow, ccol = rows // 2, cols // 2
centers = [(crow - noise_freq, ccol), (crow + noise_freq, ccol)]  # 成对
H = notch_filter(img.shape, centers, D0=5)

G = F_shift * H
G_unshift = np.fft.ifftshift(G)
img_filtered = np.abs(np.fft.ifft2(G_unshift))
```

---

### 七、一句话总结

> **陷波滤波器 = 带阻滤波器的特例 = 只抑制几个特定频率点。**
>
> - **用途**：去除周期性噪声（扫描线、网纹、摩尔纹）；
> - **找频率**：看频谱亮点、估空间周期、自动检测峰值；
> - **步骤**：变换 → 中心化 → **找亮点** → 陷波 → 反中心化 → 逆变换；
> - **关键**：频谱共轭对称，陷波要**成对处理**（否则逆变换会出现虚部，无法显示）；
> - **优点**：只抹掉几个点，对图像其他频率影响很小。
>
> 陷波是频域滤波中最实用的技术之一，专门对付周期性噪声。

### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

# ============================================================
# 1. 合成一张带周期性噪声的测试图
# ============================================================
def make_noisy_image(size=256):
    """合成测试图：灰底 + 方块 + 圆 + 水平正弦条纹噪声"""
    img = np.full((size, size), 100, dtype=np.uint8)         # 灰底
    cv2.rectangle(img, (30, 30), (100, 100), 255, -1)        # 白色方块
    cv2.circle(img, (180, 70), 35, 0, -1)                    # 黑色圆
    # 添加水平正弦条纹噪声（周期 16 像素）
    for i in range(size):
        img[i, :] = np.clip(img[i, :].astype(np.int32) + 
                            30 * np.sin(2 * np.pi * i / 16), 0, 255).astype(np.uint8)
    return img

img_noisy = make_noisy_image(256)                            # 合成带噪测试图
rows, cols = img_noisy.shape                                 # 尺寸
crow, ccol = rows // 2, cols // 2                            # 中心

# 傅里叶变换
F = np.fft.fft2(img_noisy)                                   # 傅里叶变换
F_shift = np.fft.fftshift(F)                                 # 中心化

# ============================================================
# 2. 构造陷波滤波器（成对）
# ============================================================
# 正弦条纹周期 16 → 频率 ≈ 256/16 = 16
# 噪声在频谱上表现为纵轴上的一对亮点（因为条纹是水平的）
noise_freq = 16                                              # 噪声频率
D0 = 5                                                       # 陷波半径

H = np.ones((rows, cols), np.float32)                        # 初始化全 1
# 纵轴上的对称亮点： (crow - noise_freq, ccol) 和 (crow + noise_freq, ccol)
centers = [(crow - noise_freq, ccol), (crow + noise_freq, ccol)]

for (u_k, v_k) in centers:                                   # 遍历陷波中心
    for u in range(rows):                                    # 遍历行
        for v in range(cols):                                # 遍历列
            D = np.sqrt((u - u_k)**2 + (v - v_k)**2)         # 距离
            if D <= D0:                                      # 陷波
                H[u, v] = 0

# ============================================================
# 3. 频域滤波
# ============================================================
G = F_shift * H                                              # 频域滤波
G_unshift = np.fft.ifftshift(G)                              # 反中心化
img_filtered = np.abs(np.fft.ifft2(G_unshift))               # 逆变换

# ---------- 可视化 ----------
plt.figure(figsize=(15, 10))                                 # 创建画布

plt.subplot(2, 3, 1)                                         # 子图 1
plt.title('合成测试图\n(灰底+方块+圆)')                         # 标题
plt.imshow(img_noisy, cmap='gray', vmin=0, vmax=255)         # 显示带噪图
plt.axis('off')                                              # 关闭坐标轴

plt.subplot(2, 3, 2)                                         # 子图 2
plt.title('噪声频谱\n(纵轴上有一对亮点)')                      # 标题
spec = np.log(1 + np.abs(F_shift))                           # 对数频谱
plt.imshow(spec, cmap='gray')                                # 显示频谱
plt.axis('off')                                              # 关闭坐标轴

plt.subplot(2, 3, 3)                                         # 子图 3
plt.title('陷波滤波器 H\n(在亮点处设 0)')                      # 标题
plt.imshow(H, cmap='gray', vmin=0, vmax=1)                   # 显示 H
plt.axis('off')                                              # 关闭坐标轴

plt.subplot(2, 3, 4)                                         # 子图 4
plt.title('滤波后频谱\n(亮点被抹掉)')                          # 标题
spec_filt = np.log(1 + np.abs(G))                            # 滤波后频谱
plt.imshow(spec_filt, cmap='gray')                           # 显示滤波后频谱
plt.axis('off')                                              # 关闭坐标轴

plt.subplot(2, 3, 5)                                         # 子图 5
plt.title('陷波滤波结果\n(条纹噪声被去除)')                    # 标题
plt.imshow(img_filtered, cmap='gray', vmin=0, vmax=255)      # 显示滤波结果
plt.axis('off')                                              # 关闭坐标轴

plt.tight_layout()                                           # 自动调整间距
plt.show()                                                   # 显示图像

**图像解读**

一、五张子图分别是什么

| 子图 | 内容 | 对应代码 |
|------|------|---------|
| 第 1 行第 1 列 | 合成测试图 | `img_noisy` |
| 第 1 行第 2 列 | 噪声频谱 | `np.log(1 + np.abs(F_shift))` |
| 第 1 行第 3 列 | 陷波滤波器 H | `H` |
| 第 2 行第 1 列 | 滤波后频谱 | `np.log(1 + np.abs(G))` |
| 第 2 行第 2 列 | 陷波滤波结果 | `img_filtered` |

---

二、观察到的现象

**合成测试图**

- 灰底上有白色方块、黑色圆；
- 图像上叠加了水平正弦条纹噪声，周期 16 像素。

**噪声频谱**

- 中心亮（正常低频）；
- **在纵轴上出现一对对称亮点**（周期噪声的特征）；
- 亮点位置距中心约 16 像素（对应噪声频率）。
- **为什么成对？** 因为条纹是实信号，频谱共轭对称。

**陷波滤波器 H**

- 大部分白色（保留）；
- 在纵轴上两个亮点位置各有一个黑色小圆（陷波）；
- 只抹掉噪声亮点，其他频率保留。

**滤波后频谱**

- 纵轴上的亮点被抹掉；
- 其他频率基本保留；
- 频谱看起来"干净"了。

**陷波滤波结果**

- 水平条纹**几乎完全消失**；
- 方块、圆和灰底基本保留；
- 无明显副作用。

---

三、这说明了什么

**1. 周期噪声在频谱上表现为亮点**

- 周期噪声 → 特定频率 → 频谱上成对亮点；
- 亮点位置 = 噪声频率；
- 亮点亮度 = 噪声强度。

**2. 陷波滤波精准去除周期噪声**

- 只抹掉几个亮点；
- 对图像其他频率影响很小；
- 结果：噪声消失，图像保留。

**3. 陷波的优点**

- 精准：只影响特定频率；
- 高效：几次乘法完成；
- 灵活：可处理多个噪声频率（多个陷波）。

**4. 陷波的注意事项**

- 需要人工找亮点（或自动检测）；
- 陷波半径 $D_0$ 不能太大，否则会误伤图像本身；
- **频谱共轭对称，陷波要成对处理**。

---

四、小结

> 这张图演示了陷波滤波去除周期噪声：
>
> - **周期噪声** → 频谱上一对亮点（位置由噪声频率决定）；
> - **陷波滤波器**：在亮点处设 0，其他设 1；
> - **结果**：噪声消失，图像保留；
> - **优点**：精准、高效、灵活；
> - **关键**：正确找到亮点位置，成对处理。
>
> 陷波是频域滤波中最实用的技术之一，专门对付周期性噪声。

## 7.8 章节总结与知识地图

### 一、知识地图

本章的章节结构如下（用缩进列表表示，避免框图右侧对不齐的问题）：

- **7.0 频域分析的本质**
  - 低频 = 缓慢变化；高频 = 剧烈变化
  - 傅里叶变换 = 空间域 ↔ 频域的桥梁
  - 卷积定理：空间域卷积 = 频域乘法
- **7.1 傅里叶变换工具**
  - `np.fft.fft2` / `cv2.dft`
  - `fftshift` 中心化 / `log` 显示
  - 性能对比
- **7.2 频域滤波五步流程**
  - 变换 → 中心化 → 滤波 → 反中心化 → 逆变换
  - 补零 / 向量化 / 边界效应
- **7.3 低通滤波**
  - 理想 / 巴特沃斯 / 高斯
  - 模糊、去噪
  - 振铃：理想 > 巴特沃斯 > 高斯
- **7.4 高通滤波**
  - 理想 / 巴特沃斯 / 高斯
  - 锐化、边缘检测
  - 振铃：理想 > 巴特沃斯 > 高斯
- **7.5 Laplacian 算子**
  - $H = -(u^2 + v^2)$
  - 边缘提取
  - 对噪声敏感
- **7.6 选择性滤波**
  - 带阻：抑中留外
  - 带通：留中抑外
  - $H_{BP} = 1 - H_{BR}$
- **7.7 陷波滤波**
  - 只抑几个点，精准去周期噪声
  - 三种找噪声频率方法
  - 成对处理（共轭对称）

**章节之间的依赖关系**：

- 7.0 建立直觉，7.1 提供工具，7.2 给出通用流程；
- 7.3 / 7.4 / 7.5 是三大基本滤波器，都建立在 7.2 的五步流程之上；
- 7.6 是 7.3 / 7.4 的推广（从"全通/全阻"到"选择性"）；
- 7.7 是 7.6 的特例（从"一整段"到"几个点"），也是本章最实用的技术。

### 二、核心公式速查

| 公式 | 含义 |
|------|------|
| $F(u,v) = \sum\sum f(x,y)e^{-j2\pi(ux/M+vy/N)}$ | 2D DFT |
| $f * h \Longleftrightarrow F \cdot H$ | 卷积定理 |
| $H_{LP} = \frac{1}{1+[D/D_0]^{2n}}$ | 巴特沃斯低通 |
| $H_{HP} = 1 - H_{LP}$ | 高通 = 1 - 低通 |
| $H_{Laplacian} = -(u^2+v^2)$ | Laplacian 传递函数 |
| $H_{grad} = D$ | 梯度算子传递函数 |
| $H_{BP} = 1 - H_{BR}$ | 带通 = 1 - 带阻 |
| $D(u,v) = \sqrt{(u-M/2)^2+(v-N/2)^2}$ | 到频谱中心的距离 |

### 三、关键概念对照

| 概念 | 一句话解释 |
|------|-----------|
| 低频 | 像素值变化缓慢，大结构、整体亮度 |
| 高频 | 像素值剧烈跳变，边缘、细节、噪声 |
| 中心化 | 把零频搬到频谱中心，便于设计滤波器 |
| 对数变换 | 压缩动态范围，让频谱外围也能看见 |
| 共轭对称 | 实信号频谱 $F(-u,-v) = F^*(u,v)$，滤波要成对 |
| 补零 | 避免循环卷积边界伪影，推荐补到 2 倍尺寸 |
| 振铃 | 频域硬边界 → 空间域 sinc → 波纹 |
| 传递函数 H | 频域滤波器，逐元素乘到频谱上 |
| 陷波 | 只抑几个频率点，精准去周期噪声 |

### 四、实践决策指南

**1. 什么时候用频域滤波？**

- 大核卷积（$k > 15$）；
- 去周期噪声（陷波）；
- 图像配准（相位相关）；
- 需要精确控制频率响应。

**2. 什么时候用空间域滤波？**

- 小核（3×3、5×5）；
- 非线性滤波（中值、双边）；
- 边界处理要求高；
- 实现简单优先。

**3. 滤波器怎么选？**

| 需求 | 推荐 |
|------|------|
| 平滑、去噪，最自然 | 高斯低通 |
| 平滑，过渡带可调 | 巴特沃斯低通 |
| 锐化、边缘增强，最自然 | 高斯高通 |
| 锐化，过渡带可调 | 巴特沃斯高通 |
| 边缘提取 | Laplacian / 高斯高通 |
| 去周期噪声 | 陷波（成对） |
| 提取特定频段 | 带通 |
| 抑制特定频段 | 带阻 |

### 五、常见误区

1. **❌ 只抹正频率亮点，不抹负频率** → 破坏共轭对称，逆变换变复数；
2. **❌ 频域滤波不做补零** → 循环卷积导致边界伪影；
3. **❌ 用双重循环构造滤波器** → 慢 50 倍，应向量化；
4. **❌ 认为"理想低通"是视觉效果理想** → 实际振铃严重；
5. **❌ 认为带通能精准去噪** → 带通保留一整段，会改变条纹波形，精准去噪用陷波；
6. **❌ 大图不做 FFT 尺寸优化** → OpenCV 在 2 的幂时最快。

### 六、一句话总结

> **第 7 章 = 换一个视角看图像：不看像素，看频率。**
>
> - **工具**：FFT（`np.fft` / `cv2.dft`）；
> - **流程**：变换 → 中心化 → 滤波 → 反中心化 → 逆变换；
> - **低通**：模糊、去噪；
> - **高通**：锐化、边缘；
> - **Laplacian**：$H = -(u^2+v^2)$，边缘提取；
> - **选择性**：带阻 / 带通，$H_{BP} = 1 - H_{BR}$；
> - **陷波**：精准去周期噪声；
> - **关键**：共轭对称、补零、向量化。
>
> 第 6 章空间滤波和第 7 章频域滤波，是同一件事的两种视角。

## 7.9 思考题

### 基础题

1. **为什么频谱图要中心化？** 不中心化能不能做滤波？
2. **为什么频谱显示要用对数变换？** 直接显示会怎样？
3. **低频和高频分别对应图像的什么内容？** 各举两个例子。
4. **实图像的频谱有什么对称性？** 这对滤波有什么约束？

### 进阶题

5. **只抹掉频谱上正频率的一个亮点，不抹负频率的对称亮点，逆变换结果会怎样？** 为什么？
6. **理想低通、巴特沃斯低通、高斯低通，哪个振铃最严重？** 为什么？
7. **为什么频域滤波要做补零？** 不补零会出现什么伪影？
8. **为什么用双重循环构造滤波器很慢？** 如何用 meshgrid 向量化？
9. **带通滤波结果里的条纹为什么比原图"变宽变软"？**

### 综合题

10. **一张图上有水平周期条纹噪声，如何用陷波滤波器去除？** 写出完整步骤。
11. **空间域高斯滤波和频域高斯低通，本质上是同一件事吗？** 为什么？
12. **大核卷积（如 31×31 高斯核）用空间域还是频域？** 给出复杂度分析。
13. **如何用频域方法实现 Laplacian 锐化？** 写出传递函数和步骤。

### 开放题

14. **如果图像同时有周期噪声和非周期噪声，你会怎么处理？**
15. **频域滤波和深度学习去噪，各有什么优劣？**
16. **JPEG 压缩用了 DCT（傅里叶变换的表亲），为什么不用 DFT？**